Schede concettuali prodotte da generative AI e poi passate al generatore di embedding

In [ ]:
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
TESTO_DIR = "/content/drive/MyDrive/HERO/key_results_testi"

In [ ]:
# ============================================================
# FUNZIONE: ESTRAI METADATA DAL NOME FILE
# ============================================================
def estrai_metadata(nome_file):
    base = nome_file.replace("_KeyResults.txt", "")

    # Cerca il pattern mese_anno_-_mese_anno
    periodo_match = re.search(r'([A-Za-z]{3}_\d{4})_-_([A-Za-z]{3}_\d{4})', base)

    if periodo_match:
        inizio_periodo = periodo_match.group(1).replace('_', ' ')  # es. "Apr 2017"
        fine_periodo = periodo_match.group(2).replace('_', ' ')    # es. "Sep 2017"
        periodo = f"{inizio_periodo} / {fine_periodo}"
    else:
        inizio_periodo = None
        fine_periodo = None
        periodo = base

    # Paese: tutto quello che precede il primo mese
    paese_match = re.match(r'^(.+?)_[A-Za-z]{3}_\d{4}', base)
    paese = paese_match.group(1).replace('_', ' ') if paese_match else base

    return {
        "paese": paese,
        "periodo": periodo,
        "inizio_periodo": inizio_periodo,  # es. "Apr 2017"
        "fine_periodo": fine_periodo,       # es. "Sep 2017"
        "nome_file": nome_file
    }

In [ ]:

files = [f for f in os.listdir(TESTO_DIR) if f.endswith(".txt")]
print(f"Trovati {len(files)} file...")

schede_llm = []
metadata_list = []
testi_originali = []

# Prompt ingegnerizzato per isolare i soli fattori tecnici
prompt_struttura = (
    "You are an expert humanitarian data analyst. Read the following report about food insecurity. "
    "Extract and rewrite a structured summary containing ONLY:\n"
    "1) The structural drivers of the crisis (e.g., conflict, drought, inflation, floods, displacement).\n"
    "2) Technical food security indicators (e.g., IPC phases, percentages, number of people affected).\n"
    "3) Humanitarian impacts on livelihoods and nutrition.\n\n"
    "CRITICAL CONSTRAINT: You must completely eliminate and NEVER mention any geographical location, "
    "country name, province, city, local sub-region name, or specific years/dates. Anonymous the text entirely.\n\n"
    "Report to analyze:\n"
)

print("Avvio estrazione schede cliniche tramite Gemini API...")
for nome_file in tqdm(files):
    path = os.path.join(TESTO_DIR, nome_file)
    with open(path, "r", encoding="utf-8") as f:
        testo = f.read()

    testo_pulito = re.sub(r'^.*?={40}\n\n', '', testo, flags=re.DOTALL).strip()
    if not testo_pulito:
        continue

    # Chiamata a Gemini per ripulire il testo all'istante
    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=f"{prompt_struttura}{testo_pulito}"
        )
        scheda_estratta = response.text.strip()
    except Exception as e:
        print(f"Errore sul file {nome_file}: {e}")
        continue

    schede_llm.append(scheda_estratta)
    testi_originali.append(testo_pulito) # Lo teniamo nel DF per leggerlo dopo
    metadata_list.append(estrai_metadata(nome_file))

print(f"\nGenerazione embedding sulle schede cliniche filtrate ({len(schede_llm)} documenti)...")


Trovati 497 file...
Avvio estrazione schede cliniche tramite Gemini API...


  0%|          | 1/497 [00:00<06:57,  1.19it/s]

Errore sul file Democratic_Republic_of_the_Congo_Dec_2012_-_Dec_2012_KeyResults.txt: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}


  0%|          | 2/497 [00:01<05:47,  1.42it/s]

Errore sul file Burundi_Nov_2024_-_Mar_2025_KeyResults.txt: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}


...

Errore sul file Honduras_May_2014_-_Nov_2014_KeyResults.txt: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 31.720326076s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'

 88%|████████▊ | 436/497 [00:48<00:04, 13.42it/s]

Errore sul file Haiti_Feb_2017_-_Sep_2017_KeyResults.txt: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 31.50298218s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'mode

 88%|████████▊ | 437/497 [00:48<00:06,  8.95it/s]


KeyboardInterrupt: 

In [ ]:
import os
import re
import time  # <--- CORREZIONE: Importa time per gestire la pausa
from tqdm import tqdm

files = [f for f in os.listdir(TESTO_DIR) if f.endswith(".txt")]
print(f"Trovati {len(files)} file...")

schede_llm = []
metadata_list = []
testi_originali = []

prompt_struttura = (
    "You are an expert humanitarian data analyst. Read the following report about food insecurity. "
    "Extract and rewrite a structured summary containing ONLY:\n"
    "1) The structural drivers of the crisis (e.g., conflict, drought, inflation, floods, displacement).\n"
    "2) Technical food security indicators (e.g., IPC phases, percentages, number of people affected).\n"
    "3) Humanitarian impacts on livelihoods and nutrition.\n\n"
    "CRITICAL CONSTRAINT: You must completely eliminate and NEVER mention any geographical location, "
    "country name, province, city, local sub-region name, or specific years/dates. Anonymous the text entirely.\n\n"
    "Report to analyze:\n"
)

print("Avvio estrazione schede cliniche tramite Gemini API...")
for nome_file in tqdm(files[:5]):
    path = os.path.join(TESTO_DIR, nome_file)
    with open(path, "r", encoding="utf-8") as f:
        testo = f.read()

    testo_pulito = re.sub(r'^.*?={40}\n\n', '', testo, flags=re.DOTALL).strip()
    if not testo_pulito:
        continue

    try:
        # CORREZIONE 1: Aggiornato il modello a gemini-2.0-flash
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=f"{prompt_struttura}{testo_pulito}"
        )
        scheda_estratta = response.text.strip()

        schede_llm.append(scheda_estratta)
        testi_originali.append(testo_pulito)
        metadata_list.append(estrai_metadata(nome_file))

    except Exception as e:
        print(f"Errore sul file {nome_file}: {e}")
        # Se l'errore è dovuto alla quota, attendi prima di continuare
        if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
            print("Quota superata. Pausa di recupero di 15 secondi...")
            time.sleep(15)
        continue

    # CORREZIONE 2: Pausa forzata tra i file per rispettare i limiti RPM gratuiti
    # Se il limite è di 5 richieste al minuto, una pausa di 12 secondi è l'ideale (60 / 5 = 12)
    time.sleep(12)

print(f"\nGenerazione embedding sulle schede cliniche filtrate ({len(schede_llm)} documenti)...")

Trovati 497 file...
Avvio estrazione schede cliniche tramite Gemini API...


 40%|████      | 2/5 [00:00<00:00,  4.14it/s]

Errore sul file Democratic_Republic_of_the_Congo_Dec_2012_-_Dec_2012_KeyResults.txt: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}
Errore sul file Burundi_Nov_2024_-_Mar_2025_KeyResults.txt: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}


 60%|██████    | 3/5 [00:00<00:00,  4.69it/s]

Errore sul file Madagascar_Aug_2017_-_Mar_2018_KeyResults.txt: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}
Errore sul file Yemen_Oct_2023_-_Feb_2024_KeyResults.txt: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}


100%|██████████| 5/5 [00:01<00:00,  4.71it/s]

Errore sul file Mozambique_Apr_2018_-_Sep_2018_KeyResults.txt: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}

Generazione embedding sulle schede cliniche filtrate (0 documenti)...


In [ ]:
# Usiamo il tuo modello classico (model), ma passandogli il testo filtrato dall'LLM
embeddings = model.encode(schede_cliniche_llm, normalize_embeddings=True, show_progress_bar=True)
print(f"Pipeline completata. Shape degli embedding concettuali: {embeddings.shape}")

In [ ]:
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score
import pandas as pd

# Esempio ipotetico: calcola l'impatto della geografia sui tuoi cluster
# (Sostituisci i nomi delle colonne con quelli reali del tuo DataFrame)

test_geografico = {
    "Configurazione": [
        "Modello Originale (Baseline)",
        "Opzione 1 (Instruction Embedding)",
        "Opzione 2 (Scheda Clinica LLM)"
    ],
    "Adjusted Rand Index (ARI)*": [
        adjusted_rand_score(df["paese"], df["cluster_originale"]),
        adjusted_rand_score(df["paese"], df["cluster_opzione1"]),
        adjusted_rand_score(df["paese"], df["cluster_opzione2"])
    ],
    "Adjusted Mutual Info (AMI)*": [
        adjusted_mutual_info_score(df["paese"], df["cluster_originale"]),
        adjusted_mutual_info_score(df["paese"], df["cluster_opzione1"]),
        adjusted_mutual_info_score(df["paese"], df["cluster_opzione2"])
    ]
}

df_test = pd.DataFrame(test_geografico)
print("=== VERIFICA DEL BIAS GEOGRAFICO (Più è vicino a 0, migliore è l'anonimizzazione) ===")
display(df_test)

In [ ]:
## con llama (Prima di poter eseguire il codice,
# creare un account gratuito su Hugging Face, andare sulla pagina di Llama 3.1 8B,
# accettare le condizioni di licenza e generare un Hugging Face Token dalle tue impostazioni del profilo per inserirlo nei segreti di Colab.)

# 1. Installa i pacchetti necessari (se non lo hai già fatto per Mistral)

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

model_id_llama = "meta-llama/Llama-3.1-8B-Instruct"

# 2. Carica il Tokenizer specifico di Llama
tokenizer_llama = AutoTokenizer.from_pretrained(model_id_llama)

# Se il modello non ha un token di riempimento predefinito, impostiamo quello di fine testo
if tokenizer_llama.pad_token is None:
    tokenizer_llama.pad_token = tokenizer_llama.eos_token

# 3. Carica il modello Llama ottimizzato per la GPU di Colab
model_llama = AutoModelForCausalLM.from_pretrained(
    model_id_llama,
    torch_dtype=torch.bfloat16,
    device_map="auto" # Sposta il calcolo sulla GPU automaticamente
)

# 4. Crea la pipeline di generazione testo
pipe_llama = pipeline("text-generation", model=model_llama, tokenizer=tokenizer_llama)

# 5. Configura il prompt secondo lo standard di Llama 3
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": f"{prompt_struttura}\n\nReport:\n{testo_pulito}"}
]

# 6. Genera il testo
risposta = pipe_llama(messages, max_new_tokens=500, do_sample=False)
scheda_estratta_llama = risposta[0]['generated_text'][-1]['content'].strip()
print(scheda_estratta_llama)


In [ ]:
!pip install transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.3 MB/s eta 0:00:00


In [ ]:
import os
import re
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

# 1. Configurazione del modello con quantizzazione a 4-bit per evitare i crash di Colab
model_id_llama = "meta-llama/Llama-3.1-8B-Instruct"

# Configurazione BitsAndBytes
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

print("Caricamento del tokenizer e del modello ottimizzato a 4-bit...")
tokenizer_llama = AutoTokenizer.from_pretrained(model_id_llama)

if tokenizer_llama.pad_token is None:
    tokenizer_llama.pad_token = tokenizer_llama.eos_token

model_llama = AutoModelForCausalLM.from_pretrained(
    model_id_llama,
    quantization_config=quantization_config, # <--- Applica la quantizzazione salva-memoria
    device_map="auto"
)

# pipeline di generazione testo
pipe_llama = pipeline("text-generation", model=model_llama, tokenizer=tokenizer_llama)

# 2. Configurazione cartelle e prompt
OUTPUT_CSV = "schede_estratte_llama_colab.csv"

files = [f for f in os.listdir(TESTO_DIR) if f.endswith(".txt")]
print(f"Trovati {len(files)} file...")

prompt_struttura = (
    "You are an expert humanitarian data analyst. Read the following report about food insecurity. "
    "Extract and rewrite a structured summary containing ONLY:\n"
    "1) The structural drivers of the crisis (e.g., conflict, drought, inflation, floods, displacement).\n"
    "2) Technical food security indicators (e.g., IPC phases, percentages, number of people affected).\n"
    "3) Humanitarian impacts on livelihoods and nutrition.\n\n"
    "CRITICAL CONSTRAINT: You must completely eliminate and NEVER mention any geographical location, "
    "country name, province, city, local sub-region name, or specific years/dates. Anonymous the text entirely.\n\n"
    "Report to analyze:\n"
)

print("Avvio elaborazione dei report in blocco...")

# 3. Ciclo for per elaborare tutti i file
for nome_file in tqdm(files[:5]):
    path = os.path.join(TESTO_DIR, nome_file)
    with open(path, "r", encoding="utf-8") as f:
        testo = f.read()

    testo_pulito = re.sub(r'^.*?={40}\n\n', '', testo, flags=re.DOTALL).strip()
    if not testo_pulito:
        continue

    try:
        # Configura il prompt secondo lo standard di Llama 3 (usando i tuoi messaggi)
        messages = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": f"{prompt_struttura}\n\nReport:\n{testo_pulito}"}
        ]

        # Genera il testo (nota: aggiunto clean_up_tokenization_spaces per sicurezza)
        risposta = pipe_llama(messages, max_new_tokens=500, do_sample=False)
        scheda_estratta_llama = risposta[0]['generated_text'][-1]['content'].strip()

        # Estrazione metadati e salvataggio su DataFrame
        metadata = estrai_metadata(nome_file) if 'estrai_metadata' in globals() else nome_file
        nuovo_dato = pd.DataFrame([{
            "file": nome_file,
            "metadata": metadata,
            "testo_originale": testo_pulito,
            "scheda_llm": scheda_estratta_llama
        }])

        # Scrittura incrementale sul file CSV (nessuna perdita di dati in caso di interruzione)
        nuovo_dato.to_csv(OUTPUT_CSV, mode='a', index=False, header=not os.path.exists(OUTPUT_CSV), encoding='utf-8')

    except Exception as e:
        print(f"\nErrore sul file {nome_file}: {e}")
        continue

print(f"\nProcesso completato! Il file finale è salvato in: {OUTPUT_CSV}")


Caricamento del tokenizer e del modello ottimizzato a 4-bit...


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct.
401 Client Error. (Request ID: Root=1-6a52a414-52ca81af5ecae8022abc2ee1;94b9fddf-f210-45c5-9d55-ca1aff37989b)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.1-8B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
!pip install -U bitsandbytes transformers accelerate tqdm pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.6/676.6 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 86.9 MB/s eta 0:00:00
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.3
    Uninstalling tqdm-4.67.3:
      Successfully uninstalled tqdm-4.67.3
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.1
    Uninstalling transformers-5.12.1:
      Successfully uninstalled transformers-5.12.1
ERROR: pip's dependency resolver does not cu

In [ ]:
import os
import re
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

# 1. Configurazione del modello Qwen 2.5 (Accesso libero e immediato)
model_id_qwen = "Qwen/Qwen2.5-3B-Instruct"

# Configurazione a 4-bit per garantire massima velocità sulla GPU T4 senza crash
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

print("Caricamento del Tokenizer e del modello Qwen nella GPU...")
tokenizer_qwen = AutoTokenizer.from_pretrained(model_id_qwen)

model_qwen = AutoModelForCausalLM.from_pretrained(
    model_id_qwen,
    quantization_config=quantization_config,
    device_map="auto"
)

# Creazione della pipeline ufficiale Hugging Face per la generazione del testo
pipe_qwen = pipeline("text-generation", model=model_qwen, tokenizer=tokenizer_qwen)

# 2. Configurazione dei percorsi e del prompt ingegnerizzato
OUTPUT_CSV = "schede_estratte_qwen_colab.csv"

files = [f for f in os.listdir(TESTO_DIR) if f.endswith(".txt")]
print(f"Trovati {len(files)} file da elaborare...")

prompt_struttura = (
    "You are an expert humanitarian data analyst. Read the following report about food insecurity. "
    "Extract and rewrite a structured summary containing ONLY:\n"
    "1) The structural drivers of the crisis (e.g., conflict, drought, inflation, floods, displacement).\n"
    "2) Technical food security indicators (e.g., IPC phases, percentages, number of people affected).\n"
    "3) Humanitarian impacts on livelihoods and nutrition.\n\n"
    "CRITICAL CONSTRAINT: You must completely eliminate and NEVER mention any geographical location, "
    "country name, province, city, local sub-region name, or specific years/dates. Anonymous the text entirely.\n\n"
    "Report to analyze:\n"
)

print("Avvio elaborazione globale tramite Qwen su GPU...")

# 3. Ciclo principale di estrazione sui 497 report
for nome_file in tqdm(files[:5]):
    path = os.path.join(TESTO_DIR, nome_file)
    with open(path, "r", encoding="utf-8") as f:
        testo = f.read()

    # Pulizia preliminare del file di testo
    testo_pulito = re.sub(r'^.*?={40}\n\n', '', testo, flags=re.DOTALL).strip()
    if not testo_pulito:
        continue

    try:
        # Strutturazione dei messaggi secondo lo standard chat di Qwen
        messages = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": f"{prompt_struttura}\n\nReport:\n{testo_pulito}"}
        ]

        # Generazione della risposta (do_sample=False garantisce la massima precisione)
        risposta = pipe_qwen(messages, max_new_tokens=500, do_sample=False)
        scheda_estratta_qwen = risposta[0]['generated_text'][-1]['content'].strip()

        # Estrazione metadati (utilizza la tua funzione originale se presente)
        metadata = estrai_metadata(nome_file) if 'estrai_metadata' in globals() else nome_file

        # Strutturazione in un DataFrame per il salvataggio
        nuovo_dato = pd.DataFrame([{
            "file": nome_file,
            "metadata": metadata,
            "testo_originale": testo_pulito,
            "scheda_llm": scheda_estratta_qwen
        }])

        # Scrittura incrementale: aggiunge i dati al CSV man mano che vengono estratti
        nuovo_dato.to_csv(
            OUTPUT_CSV,
            mode='a',
            index=False,
            header=not os.path.exists(OUTPUT_CSV),
            encoding='utf-8'
        )

    except Exception as e:
        print(f"\nErrore imprevisto sul file {nome_file}: {e}")
        continue

print(f"\nProcesso completato! Trovi tutti i dati pronti nel file: '{OUTPUT_CSV}'")


Caricamento del Tokenizer e del modello Qwen nella GPU...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Trovati 497 file da elaborare...
Avvio elaborazione globale tramite Qwen su GPU...


  0%|          | 0/5 [00:00<?, ?it/s][transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece token


Processo completato! Trovi tutti i dati pronti nel file: 'schede_estratte_qwen_colab.csv'


In [ ]:
df2 = pd.read_csv('schede_estratte_qwen_colab.csv')
df2['scheda_llm'][0]

'### Structural Drivers of the Crisis\n- **Conflict**: Armed conflicts in various regions of DRC, including Northern and Southern Kivu, Maniema, and Katanga.\n- **Displacement**: People expelled from Angola, Congolese repatriated, and Central African refugees.\n\n### Technical Food Security Indicators\n- **IPC Phases**: 77 regions classified in Phase 3, 8 regions in Phase 4.\n- **Household Food Consumption**: Not explicitly stated but implied to be a key indicator.\n- **Livelihoods Evolution**: Not explicitly stated but implied to be a key indicator.\n- **Nutritional State**: Children aged 6-59 months and mortality rates.\n\n### Humanitarian Impacts on Livelihoods and Nutrition\n- **Livelihoods**: Affected by armed conflicts, displacement, and recurrent shocks.\n- **Nutrition**: High infant mortality rates and malnutrition prevalent across the country, especially in Maniema province.'

In [ ]:
import os
import re
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

# 1. Configurazione del modello Qwen 2.5 (Accesso libero e immediato)
model_id_qwen = "Qwen/Qwen2.5-3B-Instruct"

# Configurazione a 4-bit per la GPU T4 di Colab (Evita i crash di memoria)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

print("Caricamento del Tokenizer e del modello Qwen nella GPU...")
tokenizer_qwen = AutoTokenizer.from_pretrained(model_id_qwen)

model_qwen = AutoModelForCausalLM.from_pretrained(
    model_id_qwen,
    quantization_config=quantization_config,
    device_map="auto"
)

# Creazione della pipeline ufficiale per la generazione del testo
pipe_qwen = pipeline("text-generation", model=model_qwen, tokenizer=tokenizer_qwen)

# 2. Configurazione dei percorsi (Verifica che la cartella sia corretta)
OUTPUT_CSV = "schede_estratte_qwen_anonime.csv"

files = [f for f in os.listdir(TESTO_DIR) if f.endswith(".txt")]
print(f"Trovati {len(files)} file da elaborare...")

# 3. Prompt di sistema e di struttura per l'anonimizzazione totale
prompt_sistema = (
    "You are an expert humanitarian data analyst. Your absolute priority is text anonymization.\n"
    "CRITICAL MANDATORY RULE: You are STRICTLY FORBIDDEN from including any real geographical names, "
    "country names, provinces, cities, regional zones, nationalities, or specific years/dates in your output.\n"
    "If the source text contains places (e.g., DRC, Kivu, Angola, Maniema, Burundi, etc.) or years, you MUST "
    "completely censor them and replace them with generic tokens like '[Country]', '[Region]', '[Province]', or '[Year]'.\n"
    "Do not make exceptions. If you fail to replace a location name, the security anonymization fails."
)

prompt_struttura = (
    "Read the following food insecurity report. Extract and rewrite a structured summary "
    "containing ONLY these three exact sections. You must replace every single specific location or year "
    "encountered with generic placeholders like [Country] or [Region]:\n\n"
    "### Structural Drivers of the Crisis\n"
    "- [Extract drivers here like conflict, drought, inflation, floods, displacement, using ONLY generic terms]\n\n"
    "### Technical Food Security Indicators\n"
    "- [Extract indicators here like IPC phases, percentages, numbers affected, without dates]\n\n"
    "### Humanitarian Impacts on Livelihoods and Nutrition\n"
    "- [Extract impacts here without mentioning specific places]\n\n"
    "Report to analyze:\n"
)

print("Avvio estrazione globale con blindatura di sicurezza su GPU...")

# 4. Ciclo principale di estrazione sui 497 report
for nome_file in tqdm(files):
    path = os.path.join(TESTO_DIR, nome_file)
    with open(path, "r", encoding="utf-8") as f:
        testo = f.read()

    # Pulizia preliminare del file di testo secondo la tua logica regex
    testo_pulito = re.sub(r'^.*?={40}\n\n', '', testo, flags=re.DOTALL).strip()
    if not testo_pulito:
        continue

    try:
        # Strutturazione dei messaggi
        messages = [
            {"role": "system", "content": prompt_sistema},
            {"role": "user", "content": f"{prompt_struttura}\n\nReport:\n{testo_pulito}"}
        ]

        # Generazione (do_sample=False assicura un comportamento rigido e deterministico)
        risposta = pipe_qwen(messages, max_new_tokens=500, do_sample=False)

        # --- BLOCCO DI ESTRAZIONE UNIVERSALE (Risolve l'errore di indice della lista/stringa) ---
        res_obj = risposta[0] if isinstance(risposta, list) else risposta
        testo_generato = res_obj['generated_text']

        if isinstance(testo_generato, list):
            # Se l'output è una lista di messaggi in formato chat
            scheda_estratta_qwen = testo_generato[-1]['content'].strip()
        else:
            # Se l'output è una stringa di testo piatta e unisce prompt + risposta
            testo_str = str(testo_generato)
            if "<|im_start|>assistant" in testo_str:
                scheda_estratta_qwen = testo_str.split("<|im_start|>assistant")[-1].replace("<|im_end|>", "").strip()
            else:
                scheda_estratta_qwen = testo_str.strip()
        # -------------------------------------------------------------------------------------

        # Estrazione metadati (utilizza la tua funzione originale se presente nel notebook)
        metadata = estrai_metadata(nome_file) if 'estrai_metadata' in globals() else nome_file

        # Strutturazione e salvataggio incrementale nel file CSV
        nuovo_dato = pd.DataFrame([{
            "file": nome_file,
            "metadata": metadata,
            "testo_originale": testo_pulito,
            "scheda_llm": scheda_estratta_qwen
        }])

        nuovo_dato.to_csv(
            OUTPUT_CSV,
            mode='a',
            index=False,
            header=not os.path.exists(OUTPUT_CSV),
            encoding='utf-8'
        )

    except Exception as e:
        print(f"\nErrore imprevisto sul file {nome_file}: {e}")
        continue

print(f"\nProcesso completato con successo! Trovi tutte le schede anonimizzate in: '{OUTPUT_CSV}'")



Caricamento del Tokenizer e del modello Qwen nella GPU...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Trovati 497 file da elaborare...
Avvio estrazione globale con blindatura di sicurezza su GPU...


  0%|          | 1/497 [00:16<2:19:38, 16.89s/it][transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
  0%|          | 2/497 [00:24<1:35:11, 11.54s/it][transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be remov

KeyboardInterrupt: 

In [ ]:
df3 = pd.read_csv('schede_estratte_qwen_anonime.csv')
df3['scheda_llm'][0]

'### Structural Drivers of the Crisis\n- [Conflict, drought, inflation, floods, displacement, using ONLY generic terms]\n\n### Technical Food Security Indicators\n- [IPC phases, percentages, numbers affected, using only generic terms]\n\n### Humanitarian Impacts on Livelihoods and Nutrition\n- [Affected areas are in need of an emergency food and agricultural assistance. Regions in humanitarian emergency situation (phase 4) are located in areas affected by armed conflicts of [Region], [Region], [Region], and [Region]. All provinces in [Country] are taken into account in phase 3. Established on the basis of a multidimensional analysis of food security, this classification is based on food security indicators which are household food consumption, livelihoods evolution, and nutritional state of children from 6-59 months and the mortality rate.]'

In [ ]:
import os
import re
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# 1. Configurazione del modello Qwen 2.5 (Float16 su GPU T4)
model_id_qwen = "Qwen/Qwen2.5-3B-Instruct"

print("Caricamento del modello nella GPU...")
tokenizer_qwen = AutoTokenizer.from_pretrained(model_id_qwen)

tokenizer_qwen.padding_side = "left"
if tokenizer_qwen.pad_token is None:
    tokenizer_qwen.pad_token = tokenizer_qwen.eos_token

model_qwen = AutoModelForCausalLM.from_pretrained(
    model_id_qwen,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Creazione della pipeline con batching ottimizzato
pipe_qwen = pipeline(
    "text-generation",
    model=model_qwen,
    tokenizer=tokenizer_qwen,
    batch_size=8
)

# 2. Configurazione dei percorsi

OUTPUT_CSV = "schede_estratte_qwen_anonimo_fluido.csv"

files = [f for f in os.listdir(TESTO_DIR) if f.endswith(".txt")]
print(f"Trovati {len(files)} file da elaborare...")

# Prompt per l'anonimizzazione senza placeholder strutturali
prompt_sistema = (
    "You are an expert humanitarian data analyst. Your job is to rewrite food insecurity reports into an anonymous summary.\n"
    "CRITICAL RULE 1: Absolute Anonymization. You must completely eliminate all names of countries, cities, regions, provinces, or dates.\n"
    "CRITICAL RULE 2: No Placeholders. Do NOT use tags like '[Country]', '[Region]', '[Province]', or '[Year]'. "
    "Instead, use generic descriptions or passive verbs (e.g., write 'the affected areas', 'the population', 'recurrent shocks' instead of naming places).\n"
    "CRITICAL RULE 3: No Formatting Labels. Do NOT write section headers, titles, numbers, bullet points, or markdown signs (like ###). "
    "Write ONLY one single paragraph of continuous, natural, and clean text."
)

prompt_struttura = (
    "Analyze the following report and write a single, fluid paragraph. "
    "Combine the structural drivers, technical food security indicators (including IPC phases and numbers), "
    "and humanitarian impacts into one smooth narrative.\n"
    "Remember: No titles, no location names, no years, and no bracketed placeholders. Write only plain text.\n\n"
    "Report to analyze:\n"
)

# 3. Preparazione dei testi
lista_messaggi = []
lista_metadata = []
lista_nomi_file = []
lista_testi_puliti = []

print("Fase 1: Lettura e pulizia dei file...")
for nome_file in tqdm(files[:5]):
    path = os.path.join(TESTO_DIR, nome_file)
    with open(path, "r", encoding="utf-8") as f:
        testo = f.read()

    testo_pulito = re.sub(r'^.*?={40}\n\n', '', testo, flags=re.DOTALL).strip()
    if not testo_pulito:
        continue

    messages = [
        {"role": "system", "content": prompt_sistema},
        {"role": "user", "content": f"{prompt_struttura}\n\nReport:\n{testo_pulito}"}
    ]

    lista_messaggi.append(messages)
    lista_nomi_file.append(nome_file)
    lista_testi_puliti.append(testo_pulito)
    metadata = estrai_metadata(nome_file) if 'estrai_metadata' in globals() else nome_file
    lista_metadata.append(metadata)

print("Fase 2: Generazione del testo anonimo e fluido su GPU...")
risultati_finali = []

# --- CORREZIONE CRITICA: Gestione corretta dell'output del generatore di batch ---
for i, risposte_batch in enumerate(tqdm(pipe_qwen(lista_messaggi, max_new_tokens=500, do_sample=False), total=len(lista_messaggi))):
    try:
        # Quando usiamo la pipeline in questo modo, risposte_batch è una lista contenente un dizionario
        passaggio_testo = risposte_batch[0]['generated_text'] if isinstance(risposte_batch, list) else risposte_batch['generated_text']

        if isinstance(passaggio_testo, list):
            # Se la pipeline restituisce la cronologia chat completa
            scheda_estratta = passaggio_testo[-1]['content'].strip()
        else:
            # Se restituisce una stringa piatta (Prompt + Risposta concatenati)
            testo_str = str(passaggio_testo)
            if "<|im_start|>assistant" in testo_str:
                scheda_estratta = testo_str.split("<|im_start|>assistant")[-1].replace("<|im_end|>", "").strip()
            else:
                scheda_estratta = testo_str.strip()

        # Rimuove eventuali residui testuali che citano il prompt di esempio
        scheda_estratta = re.sub(r'\[.*?\]', '', scheda_estratta).strip()

        risultati_finali.append({
            "file": lista_nomi_file[i],
            "metadata": lista_metadata[i],
            "testo_originale": lista_testi_puliti[i],
            "scheda_llm": scheda_estratta
        })

        # Salvataggio continuativo sul file CSV ogni 20 documenti
        if i % 20 == 0:
            pd.DataFrame(risultati_finali).to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

    except Exception as e:
        print(f"\nErrore sull'indice {i}: {e}")
        continue

# Salvataggio finale
df_finale = pd.DataFrame(risultati_finali)
df_finale.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
print(f"\nProcesso completato! Il file finale pulito è: '{OUTPUT_CSV}'")



Caricamento del modello nella GPU...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Trovati 497 file da elaborare...
Fase 1: Lettura e pulizia dei file...


100%|██████████| 5/5 [00:00<00:00, 176.51it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Fase 2: Generazione del testo anonimo e fluido su GPU...


[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
100%|██████████| 5/5 [00:00<00:00, 80.01it/s]


Processo completato! Il file finale pulito è: 'schede_estratte_qwen_anonimo_fluido.csv'


In [ ]:
df4 = pd.read_csv('schede_estratte_qwen_anonimo_fluido.csv')
df4['scheda_llm'][0]


'In December 2012, an analysis using the Integrated Food Security Classification Framework (IPC) revealed that 6.4 million people across the Democratic Republic of Congo (DRC) faced severe food and livelihood crises, with 77 regions categorized in Phase 3 and 8 in Phase 4. These areas require urgent emergency food and agricultural assistance. Phases 3 and 4 were predominantly found in conflict-affected regions including Northern and Southern Kivu, Maniema, and Katanga provinces. Compared to the 7th IPC analysis in June 2012, there was an overall increase of 17% in affected individuals, with more regions now in Phase 3 (up from 63 to 77) and 8 in Phase 4 (up from 3 to 8). The analysis highlighted four distinct scenarios: regions experiencing resurgence of conflicts and violence, those impacted by people expelled from Angola, Congolese repatriated, and Central African refugees; landlocked regions facing chronic poverty and malnutrition; and the entire country grappling with persistent fo

In [ ]:
df4['testo_originale'][0]

'The 8th analysis cycle on the Integrated Food Security Classification Framework (IPC) of DRC held in December 2012 identified 6.4 million people affected by a situation of food and livelihood crises, 77 regions have been classified in phase 3 and 8 regions in Phase 4 throughout DRC. The affected areas are in need of an emergency food and agricultural assistance. Regions in humanitarian emergency situation  (phase 4) are located in areas affected by armed conflicts of Northern Kivu (Rutshuru, Masisi), of Southern Kivu (Kalehe, Shabunda), of Maniema (Pangi) and Katanga (Mitwaba, Manono, Pweto), whereas all the provinces in DRC  are  taken  into  account  in  phase  3.  Established on  the  basis  of  a multidimensional  analysis  of  food  security,  this  classification  is  based  on  food security indicators which are household food consumption, livelihoods evolution, and nutritional state of children from 6-59 months and the mortality rate.\nAs  compared  to  the  7th IPC  analysis 

In [ ]:
##con mistral

# 1. Installa le librerie necessarie (esegui in una cella separata)
!pip install transformers accelerate bitsandbytes

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

model_id = "mistralai/Mistral-7B-Instruct-v0.3"

# 2. Carica il tokenizer e il modello ottimizzato per occupare pochissima RAM
tokenizer = AutoTokenizer.from_pretrained(model_id)
model_mistral = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto" # Distribuisce automaticamente il modello sulla GPU
)

# 3. Crea la pipeline di generazione testo
pipe = pipeline("text-generation", model=model_mistral, tokenizer=tokenizer)

# 4. Struttura il messaggio seguendo il formato speciale richiesto da Mistral [INST] [/INST]
prompt_completo = f"<s>[INST] {prompt_struttura} \n\n Report: {testo_pulito} [/INST]"

# 5. Genera la scheda clinica
risposta = pipe(prompt_completo, max_new_tokens=500, do_sample=False)
scheda_estratta = risposta[0]['generated_text'].split("[/INST]")[-1].strip()
print(scheda_estratta)


In [ ]:
from huggingface_hub import login
login()

In [ ]:
import os
import re
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

# 1. Configurazione di Llama 3.1 8B ottimizzato a 4-bit (Sfrutta la tua licenza approvata)
model_id_llama = "meta-llama/Llama-3.2-3B"

# Configurazione BitsAndBytes per far entrare l'8B nella GPU T4 senza Out-Of-Memory
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

print("Caricamento del Tokenizer e del modello Llama nella GPU...")
tokenizer_llama = AutoTokenizer.from_pretrained(model_id_llama)

if tokenizer_llama.pad_token is None:
    tokenizer_llama.pad_token = tokenizer_llama.eos_token

model_llama = AutoModelForCausalLM.from_pretrained(
    model_id_llama,
    quantization_config=quantization_config,
    device_map="auto"
)

# Creazione della pipeline ufficiale di generazione testo
pipe_llama = pipeline("text-generation", model=model_llama, tokenizer=tokenizer_llama)

# 2. Configurazione dei percorsi di Colab
OUTPUT_CSV = "schede_estratte_llama_anonime.csv"

files = [f for f in os.listdir(TESTO_DIR) if f.endswith(".txt")]
print(f"Trovati {len(files)} file da elaborare...")

# --- ARCHITETTURA PROMPT A DUE FASI PER LA MASSIMA SICUREZZA ---
# Passo 1: Estrazione pura dei dati complessi dal testo grezzo
prompt_estrazione = (
    "Analyze the following humanitarian report. Extract a short, clean summary in one paragraph containing:\n"
    "1) The structural drivers (conflict, displacement, inflation).\n"
    "2) Technical food security indicators (IPC phases, numbers affected, percentages).\n"
    "3) Humanitarian impacts on livelihoods and nutrition.\n\n"
    "Report to analyze:\n"
)

# Passo 2: Riscrittura e oscuramento totale dei riferimenti spazio-temporali
prompt_anonimizzazione = (
    "You are a strict data redaction bot. Your only job is to rewrite the text provided by the user to make it completely anonymous.\n"
    "CRITICAL RULES:\n"
    "1) Delete all country names, city names, provinces, continents, regions, or nationalities.\n"
    "2) Delete all specific years, months, or dates.\n"
    "3) Rewrite the sentences so the text flows naturally in a single paragraph without leaving placeholders like [Country] or [Region]. "
    "Use phrases like 'the affected areas', 'the population', 'neighboring territories', or 'recent periods'.\n"
    "4) Do not include headers, titles, or bullet points.\n\n"
    "Text to anonymize:\n"
)

print("Avvio elaborazione sequenziale su GPU...")
risultati_finali = []

# Elaborazione controllata dei file
for i, nome_file in enumerate(tqdm(files[:5])):
    path = os.path.join(TESTO_DIR, nome_file)
    with open(path, "r", encoding="utf-8") as f:
        testo = f.read()

    # Pulizia regex nativa dei report
    testo_pulito = re.sub(r'^.*?={40}\n\n', '', testo, flags=re.DOTALL).strip()
    if not testo_pulito:
        continue

    try:
        # FASE 1: Generazione dell'estratto tecnico grezzo
        messages_fase1 = [
            {"role": "system", "content": "You are a helpful assistant that extracts raw text data."},
            {"role": "user", "content": f"{prompt_estrazione}\n\nReport:\n{testo_pulito}"}
        ]

        risposta_fase1 = pipe_llama(messages_fase1, max_new_tokens=400, do_sample=False)
        estratto_grezzo = risposta_fase1[0]['generated_text'][-1]['content'].strip()

        # FASE 2: Anonimizzazione fluida applicata sull'estratto corto della Fase 1
        messages_fase2 = [
            {"role": "system", "content": "You are a strict data anonymization tool. You only output plain anonymous text."},
            {"role": "user", "content": f"{prompt_anonimizzazione}{estratto_grezzo}"}
        ]

        risposta_fase2 = pipe_llama(messages_fase2, max_new_tokens=400, do_sample=False)
        scheda_anonima = risposta_fase2[0]['generated_text'][-1]['content'].strip()

        # Pulizia finale di sicurezza da tag residui
        scheda_anonima = re.sub(r'\[.*?\]', '', scheda_anonima).strip()
        scheda_anonima = re.sub(r'###.*?\n', '', scheda_anonima).strip()

        # Funzione metadati nativa
        metadata = estrai_metadata(nome_file) if 'estrai_metadata' in globals() else nome_file

        # Salvataggio incrementale nel dataframe e nel CSV
        nuovo_dato = pd.DataFrame([{
            "file": nome_file,
            "metadata": metadata,
            "testo_originale": testo_pulito,
            "scheda_llm": scheda_anonima
        }])

        nuovo_dato.to_csv(
            OUTPUT_CSV,
            mode='a',
            index=False,
            header=not os.path.exists(OUTPUT_CSV),
            encoding='utf-8'
        )

    except Exception as e:
        print(f"\nErrore imprevisto sul file {nome_file}: {e}")
        continue

print(f"\nProcesso completato! Trovi tutte le schede anonimizzate in: '{OUTPUT_CSV}'")


Caricamento del Tokenizer e del modello Llama nella GPU...


config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Trovati 497 file da elaborare...
Avvio elaborazione sequenziale su GPU...


100%|██████████| 5/5 [00:00<00:00, 274.81it/s]


Errore imprevisto sul file Democratic_Republic_of_the_Congo_Dec_2012_-_Dec_2012_KeyResults.txt: Cannot use chat template functions because tokenizer.chat_template is not set and no template argument was passed! For information about writing templates and setting the tokenizer.chat_template attribute, please see the documentation at https://huggingface.co/docs/transformers/main/en/chat_templating

Errore imprevisto sul file Burundi_Nov_2024_-_Mar_2025_KeyResults.txt: Cannot use chat template functions because tokenizer.chat_template is not set and no template argument was passed! For information about writing templates and setting the tokenizer.chat_template attribute, please see the documentation at https://huggingface.co/docs/transformers/main/en/chat_templating

Errore imprevisto sul file Madagascar_Aug_2017_-_Mar_2018_KeyResults.txt: Cannot use chat template functions because tokenizer.chat_template is not set and no template argument was passed! For information about writing templ

In [ ]:
!pip install groq pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.2 MB/s eta 0:00:00


In [ ]:
import os
import re
import time
import pandas as pd
from tqdm import tqdm
from groq import Groq

# 1. Configurazione del client Groq (Incolla qui la tua API Key di Groq)

client = Groq(api_key=GROQ_API_KEY)

# Usiamo Llama 3.3 70B: precisissimo sull'anonimizzazione fluida
MODELLO_LLAMA = "llama-3.3-70b-versatile"

OUTPUT_CSV = "schede_estratte_groq_llama.csv"

files = [f for f in os.listdir(TESTO_DIR) if f.endswith(".txt")]
print(f"Trovati {len(files)} file da elaborare...")

# Prompt blindati per testo fluido senza etichette o rimasugli geografici
prompt_sistema = (
    "You are an expert humanitarian data analyst. Your job is to rewrite food insecurity reports into an anonymous summary.\n"
    "CRITICAL RULE 1: Absolute Anonymization. You must completely eliminate all names of countries, cities, regions, provinces, or dates.\n"
    "CRITICAL RULE 2: No Placeholders. Do NOT use tags like '[Country]', '[Region]', '[Province]', or '[Year]'. "
    "Instead, use generic descriptions or passive verbs (e.g., write 'the affected areas', 'the population', 'recurrent shocks' instead of naming places).\n"
    "CRITICAL RULE 3: No Formatting Labels. Do NOT write section headers, titles, numbers, bullet points, or markdown signs (like ###). "
    "Write ONLY one single paragraph of continuous, natural, and clean text."
)

prompt_struttura = (
    "Analyze the following report and write a single, fluid paragraph. "
    "Combine the structural drivers, technical food security indicators (including IPC phases and numbers), "
    "and humanitarian impacts into one smooth narrative.\n"
    "Remember: No titles, no location names, no years, and no bracketed placeholders. Write only plain text.\n\n"
    "Report to analyze:\n"
)

print("Avvio estrazione tramite Llama via Groq API...")
risultati_finali = []

for nome_file in tqdm(files[:5]):
    path = os.path.join(TESTO_DIR, nome_file)
    with open(path, "r", encoding="utf-8") as f:
        testo = f.read()

    testo_pulito = re.sub(r'^.*?={40}\n\n', '', testo, flags=re.DOTALL).strip()
    if not testo_pulito:
        continue

    successo = False
    tentativi = 0
    scheda_anonima = ""

    # Gestione automatica dei limiti di token al minuto (TPM) del piano free di Groq
    while not successo and tentativi < 5:
        try:
            completion = client.chat.completions.create(
                model=MODELLO_LLAMA,
                messages=[
                    {"role": "system", "content": prompt_sistema},
                    {"role": "user", "content": f"{prompt_struttura}\n\nReport:\n{testo_pulito}"}
                ],
                temperature=0.1,
                max_tokens=500
            )
            scheda_anonima = completion.choices.message.content.strip()
            successo = True

        except Exception as e:
            errore_str = str(e)
            if "429" in errore_str or "rate_limit_exceeded" in errore_str or "RESOURCE_EXHAUSTED" in errore_str:
                print(f"\n[Freno di Sicurezza] Limite di Token raggiunto sul file {nome_file}. Attendo 65 secondi...")
                time.sleep(65)
                tentativi += 1
            else:
                print(f"\nErrore imprevisto sul file {nome_file}: {e}")
                tentativi += 1
                time.sleep(5)

    if successo:
        # Rimozione di sicurezza di eventuali scampoli di parentesi quadre o titoli generati per sbaglio
        scheda_anonima = re.sub(r'\[.*?\]', '', scheda_anonima).strip()
        scheda_anonima = re.sub(r'###.*?\n', '', scheda_anonima).strip()

        metadata = estrai_metadata(nome_file) if 'estrai_metadata' in globals() else nome_file

        # Salvataggio incrementale nel CSV (nessuna perdita di dati se si interrompe)
        nuovo_dato = pd.DataFrame([{
            "file": nome_file,
            "metadata": metadata,
            "testo_originale": testo_pulito,
            "scheda_llm": scheda_anonima
        }])

        nuovo_dato.to_csv(OUTPUT_CSV, mode='a', index=False, header=not os.path.exists(OUTPUT_CSV), encoding='utf-8')

        # Pausa preventiva di 10 secondi per distribuire i token ed evitare di attivare continuamente il freno di sicurezza
        time.sleep(10)
    else:
        print(f"\n[Salto File] Impossibile elaborare {nome_file} dopo diversi tentativi.")

print(f"\nProcesso completato! Il file finale pulito è salvato in: '{OUTPUT_CSV}'")


Trovati 497 file da elaborare...
Avvio estrazione tramite Llama via Groq API...


  0%|          | 0/5 [00:00<?, ?it/s]


Errore imprevisto sul file Democratic_Republic_of_the_Congo_Dec_2012_-_Dec_2012_KeyResults.txt: 'list' object has no attribute 'message'

Errore imprevisto sul file Democratic_Republic_of_the_Congo_Dec_2012_-_Dec_2012_KeyResults.txt: 'list' object has no attribute 'message'

Errore imprevisto sul file Democratic_Republic_of_the_Congo_Dec_2012_-_Dec_2012_KeyResults.txt: 'list' object has no attribute 'message'

Errore imprevisto sul file Democratic_Republic_of_the_Congo_Dec_2012_-_Dec_2012_KeyResults.txt: 'list' object has no attribute 'message'

Errore imprevisto sul file Democratic_Republic_of_the_Congo_Dec_2012_-_Dec_2012_KeyResults.txt: 'list' object has no attribute 'message'


 20%|██        | 1/5 [00:31<02:05, 31.36s/it]


[Salto File] Impossibile elaborare Democratic_Republic_of_the_Congo_Dec_2012_-_Dec_2012_KeyResults.txt dopo diversi tentativi.

Errore imprevisto sul file Burundi_Nov_2024_-_Mar_2025_KeyResults.txt: 'list' object has no attribute 'message'

Errore imprevisto sul file Burundi_Nov_2024_-_Mar_2025_KeyResults.txt: 'list' object has no attribute 'message'

Errore imprevisto sul file Burundi_Nov_2024_-_Mar_2025_KeyResults.txt: 'list' object has no attribute 'message'

Errore imprevisto sul file Burundi_Nov_2024_-_Mar_2025_KeyResults.txt: 'list' object has no attribute 'message'

Errore imprevisto sul file Burundi_Nov_2024_-_Mar_2025_KeyResults.txt: 'list' object has no attribute 'message'


...

 60%|██████    | 3/5 [01:32<01:01, 30.84s/it]


[Salto File] Impossibile elaborare Madagascar_Aug_2017_-_Mar_2018_KeyResults.txt dopo diversi tentativi.

Errore imprevisto sul file Yemen_Oct_2023_-_Feb_2024_KeyResults.txt: 'list' object has no attribute 'message'

Errore imprevisto sul file Yemen_Oct_2023_-_Feb_2024_KeyResults.txt: 'list' object has no attribute 'message'

Errore imprevisto sul file Yemen_Oct_2023_-_Feb_2024_KeyResults.txt: 'list' object has no attribute 'message'

Errore imprevisto sul file Yemen_Oct_2023_-_Feb_2024_KeyResults.txt: 'list' object has no attribute 'message'


 60%|██████    | 3/5 [01:58<01:19, 39.59s/it]


Errore imprevisto sul file Yemen_Oct_2023_-_Feb_2024_KeyResults.txt: 'list' object has no attribute 'message'


KeyboardInterrupt: 

In [ ]:
import os
import re
import time
import pandas as pd
from tqdm import tqdm
from groq import Groq

# 1. Configurazione del client Groq (Incolla qui la tua API Key di Groq)
client = Groq(api_key=GROQ_API_KEY)

# Usiamo Llama 3.3 70B: incredibilmente potente e preciso sui vincoli logici
MODELLO_LLAMA = "llama-3.3-70b-versatile"

OUTPUT_CSV = "schede_estratte_groq_llama.csv"

files = [f for f in os.listdir(TESTO_DIR) if f.endswith(".txt")]
print(f"Trovati {len(files)} file da elaborare...")

# Prompt blindati per testo fluido senza etichette o rimasugli geografici
prompt_sistema = (
    "You are an expert humanitarian data analyst. Your job is to rewrite food insecurity reports into an anonymous summary.\n"
    "CRITICAL RULE 1: Absolute Anonymization. You must completely eliminate all names of countries, cities, regions, provinces, or dates.\n"
    "CRITICAL RULE 2: No Placeholders. Do NOT use tags like '[Country]', '[Region]', '[Province]', or '[Year]'. "
    "Instead, use generic descriptions or passive verbs (e.g., write 'the affected areas', 'the population', 'recurrent shocks' instead of naming places).\n"
    "CRITICAL RULE 3: No Formatting Labels. Do NOT write section headers, titles, numbers, bullet points, or markdown signs (like ###). "
    "Write ONLY one single paragraph of continuous, natural, and clean text."
)

prompt_struttura = (
    "Analyze the following report and write a single, fluid paragraph. "
    "Combine the structural drivers, technical food security indicators (including IPC phases and numbers), "
    "and humanitarian impacts into one smooth narrative.\n"
    "Remember: No titles, no location names, no years, and no bracketed placeholders. Write only plain text.\n\n"
    "Report to analyze:\n"
)

print("Avvio estrazione tramite Llama via Groq API...")
risultati_finali = []

# Ciclo principale di elaborazione
for nome_file in tqdm(files[:5]):
    path = os.path.join(TESTO_DIR, nome_file)
    with open(path, "r", encoding="utf-8") as f:
        testo = f.read()

    # Pulizia del testo secondo la tua regex nativa
    testo_pulito = re.sub(r'^.*?={40}\n\n', '', testo, flags=re.DOTALL).strip()
    if not testo_pulito:
        continue

    successo = False
    tentativi = 0
    scheda_anonima = ""

    # Gestione automatica dei limiti di token al minuto (TPM) del piano free di Groq
    while not successo and tentativi < 5:
        try:
            completion = client.chat.completions.create(
                model=MODELLO_LLAMA,
                messages=[
                    {"role": "system", "content": prompt_sistema},
                    {"role": "user", "content": f"{prompt_struttura}\n\nReport:\n{testo_pulito}"}
                ],
                temperature=0.1,
                max_tokens=500
            )

            # --- ESTRAZIONE STANDARD CORRETTA E SICURA ---
            scheda_anonima = completion.choices[0].message.content.strip()
            successo = True

        except Exception as e:
            errore_str = str(e)
            # Rilevamento dei limiti di velocità (Quota API esaurita al minuto)
            if "429" in errore_str or "rate_limit_exceeded" in errore_str or "RESOURCE_EXHAUSTED" in errore_str:
                print(f"\n[Freno di Sicurezza] Limite di Token raggiunto sul file {nome_file}. Attendo 65 secondi...")
                time.sleep(65)
                tentativi += 1
            else:
                print(f"\nErrore imprevisto sul file {nome_file}: {e}")
                tentativi += 1
                time.sleep(5)

    if successo:
        # Rimozione di sicurezza finale di eventuali anomalie grafiche (es. parentesi quadre)
        scheda_anonima = re.sub(r'\[.*?\]', '', scheda_anonima).strip()
        scheda_anonima = re.sub(r'###.*?\n', '', scheda_anonima).strip()

        # Gestione metadati nativa
        metadata = estrai_metadata(nome_file) if 'estrai_metadata' in globals() else nome_file

        # Salvataggio incrementale nel CSV ad ogni iterazione
        nuovo_dato = pd.DataFrame([{
            "file": nome_file,
            "metadata": metadata,
            "testo_originale": testo_pulito,
            "scheda_llm": scheda_anonima
        }])

        nuovo_dato.to_csv(OUTPUT_CSV, mode='a', index=False, header=not os.path.exists(OUTPUT_CSV), encoding='utf-8')

        # Pausa preventiva obbligatoria (10 secondi) per distribuire i token nel piano gratuito di Groq
        time.sleep(10)
    else:
        print(f"\n[Salto File] Impossibile elaborare {nome_file} dopo 5 tentativi.")

print(f"\nProcesso completato! Il file finale pulito è salvato in: '{OUTPUT_CSV}'")


Trovati 497 file da elaborare...
Avvio estrazione tramite Llama via Groq API...


100%|██████████| 5/5 [00:55<00:00, 11.14s/it]


Processo completato! Il file finale pulito è salvato in: 'schede_estratte_groq_llama.csv'


In [ ]:
df_groq = pd.read_csv('/content/schede_estratte_groq_llama.csv')
df_groq['scheda_llm'][0]

'The population is experiencing a significant level of food and livelihood crises, with a substantial number of people in need of emergency food and agricultural assistance, due to various structural drivers and recurrent shocks that have led to a deterioration in their living conditions. A multidimensional analysis of food security indicators, including household food consumption, livelihoods evolution, and nutritional state of children, has revealed that a large portion of the affected areas are classified in phase 3, indicating a crisis situation, while some areas are in a humanitarian emergency situation, classified in phase 4, where the effects of armed conflicts and other types of violence have resulted in severe food insecurity. The number of people affected by food and livelihood crises has increased, with more regions being affected compared to previous periods, and the geographical dimension of the crisis has expanded, with zones affected by armed conflicts being particularly

In [ ]:
df_groq['testo_originale'][0]

'The 8th analysis cycle on the Integrated Food Security Classification Framework (IPC) of DRC held in December 2012 identified 6.4 million people affected by a situation of food and livelihood crises, 77 regions have been classified in phase 3 and 8 regions in Phase 4 throughout DRC. The affected areas are in need of an emergency food and agricultural assistance. Regions in humanitarian emergency situation  (phase 4) are located in areas affected by armed conflicts of Northern Kivu (Rutshuru, Masisi), of Southern Kivu (Kalehe, Shabunda), of Maniema (Pangi) and Katanga (Mitwaba, Manono, Pweto), whereas all the provinces in DRC  are  taken  into  account  in  phase  3.  Established on  the  basis  of  a multidimensional  analysis  of  food  security,  this  classification  is  based  on  food security indicators which are household food consumption, livelihoods evolution, and nutritional state of children from 6-59 months and the mortality rate.\nAs  compared  to  the  7th IPC  analysis 

In [ ]:
df_groq['testo_originale'][1]

'Between January and March 2025, which coincides with the harvest period, nearly 1.2 million people (10 percent of the total population analysed) are projected to be in IPC Phase 3 (Crisis). This is a marked improvement from the current period (November to December 2024), where 1.9 million people were classified in IPC Phase 3 or above (Crisis or worse). \nThe improvement is likely a result of the expected favourable agricultural performance and abundant rainfall, as well as the increase in household food stocks, even if during this period, the prices of manufactured products are expected to remain higher than average due to the high cost transportation.'

In [ ]:
df_groq['scheda_llm'][1]

'The population is experiencing a period of relative improvement in food security, with a notable decline in the number of people facing severe food insecurity, from nearly 1.9 million to approximately 1.2 million, which represents about 10 percent of the total population, and are projected to be in a state of crisis, as indicated by technical food security indicators, during the harvest period, which is likely attributed to favourable agricultural performance and abundant rainfall, as well as an increase in household food stocks, although the prices of essential items are expected to remain elevated due to high transportation costs, which is a recurrent shock that affects the overall well-being of the affected areas, where the combination of structural drivers and technical indicators suggests a complex and dynamic situation, with humanitarian impacts that are still being felt, despite the slight improvement, highlighting the need for continued support and monitoring to address the un

In [ ]:
import os
import re
import time
import pandas as pd
from tqdm import tqdm
from groq import Groq

# 1. Configurazione del client Groq (Incolla qui la tua API Key di Groq)

client = Groq(api_key=GROQ_API_KEY)

# Usiamo Llama 3.3 70B: incredibilmente potente e preciso sui vincoli logici
MODELLO_LLAMA = "llama-3.3-70b-versatile"

OUTPUT_CSV = "schede_estratte_groq_llama.csv"

files = [f for f in os.listdir(TESTO_DIR) if f.endswith(".txt")]
print(f"Trovati {len(files)} file da elaborare...")

# Prompt blindati per testo fluido senza etichette o rimasugli geografici
# PROMPT DI SISTEMA: Separa Current da Projected, mantiene i numeri, vieta i titoli
prompt_sistema = (
    "You are an expert humanitarian data analyst. Your job is to rewrite food insecurity reports into an anonymous summary.\n"
    "CRITICAL RULE 1: Absolute Anonymization. You must completely eliminate all names of countries, cities, regions, provinces, or dates (months/years).\n"
    "CRITICAL RULE 2: No Placeholders. Do NOT use tags like '[Country]' or '[Region]'. Use generic words like 'the affected areas', 'the population', or 'the country'.\n"
    "CRITICAL RULE 3: Exact Data Integrity. You MUST preserve all exact numbers, percentages, figures, and technical classifications (e.g., IPC Phase 3, IPC Phase 4, 1.2 million, 10 percent). Do NOT convert numbers into generic words.\n"
    "CRITICAL RULE 4: Strict Period Separation. You must clearly and strictly separate current data from projected data. Never blend them into the same sentence.\n"
    "CRITICAL RULE 5: No Section Headers. Do NOT write markdown titles (like ###), labels, or headers. Write ONLY a plain bulleted list of exactly 4 simple points, without any introductory text."
)

prompt_struttura = (
    "Analyze the following report and write a clear summary using exactly 4 plain bullet points (without any title or label above them). "
    "Maintain every single numerical figure, percentage, and IPC phase exactly as written, but anonymize all places and dates:\n"
    "- Point 1: The structural drivers, agricultural performance, and economic factors (like prices or transportation costs).\n"
    "- Point 2: The exact technical indicators for the CURRENT period (numbers of people, percentages, and IPC phases).\n"
    "- Point 3: The exact technical indicators for the PROJECTED period (numbers of people, percentages, and IPC phases).\n"
    "- Point 4: The humanitarian impacts on the livelihoods and nutrition of the population.\n\n"
    "Remember: Keep all numbers, separate current from projected, remove all location names/dates, and do not use headers. Write only plain text bullets.\n\n"
    "Report to analyze:\n"
)


print("Avvio estrazione tramite Llama via Groq API...")
risultati_finali = []

# Ciclo principale di elaborazione
for nome_file in tqdm(files[:5]):
    path = os.path.join(TESTO_DIR, nome_file)
    with open(path, "r", encoding="utf-8") as f:
        testo = f.read()

    # Pulizia del testo secondo la tua regex nativa
    testo_pulito = re.sub(r'^.*?={40}\n\n', '', testo, flags=re.DOTALL).strip()
    if not testo_pulito:
        continue

    successo = False
    tentativi = 0
    scheda_anonima = ""

    # Gestione automatica dei limiti di token al minuto (TPM) del piano free di Groq
    while not successo and tentativi < 5:
        try:
            completion = client.chat.completions.create(
                model=MODELLO_LLAMA,
                messages=[
                    {"role": "system", "content": prompt_sistema},
                    {"role": "user", "content": f"{prompt_struttura}\n\nReport:\n{testo_pulito}"}
                ],
                temperature=0.1,
                max_tokens=500
            )

            # --- ESTRAZIONE STANDARD CORRETTA E SICURA ---
            scheda_anonima = completion.choices[0].message.content.strip()
            successo = True

        except Exception as e:
            errore_str = str(e)
            # Rilevamento dei limiti di velocità (Quota API esaurita al minuto)
            if "429" in errore_str or "rate_limit_exceeded" in errore_str or "RESOURCE_EXHAUSTED" in errore_str:
                print(f"\n[Freno di Sicurezza] Limite di Token raggiunto sul file {nome_file}. Attendo 65 secondi...")
                time.sleep(65)
                tentativi += 1
            else:
                print(f"\nErrore imprevisto sul file {nome_file}: {e}")
                tentativi += 1
                time.sleep(5)

    if successo:
        # Rimozione di sicurezza finale di eventuali anomalie grafiche (es. parentesi quadre)
        scheda_anonima = re.sub(r'\[.*?\]', '', scheda_anonima).strip()
        scheda_anonima = re.sub(r'###.*?\n', '', scheda_anonima).strip()

        # Gestione metadati nativa
        metadata = estrai_metadata(nome_file) if 'estrai_metadata' in globals() else nome_file

        # Salvataggio incrementale nel CSV ad ogni iterazione
        nuovo_dato = pd.DataFrame([{
            "file": nome_file,
            "metadata": metadata,
            "testo_originale": testo_pulito,
            "scheda_llm": scheda_anonima
        }])

        nuovo_dato.to_csv(OUTPUT_CSV, mode='a', index=False, header=not os.path.exists(OUTPUT_CSV), encoding='utf-8')

        # Pausa preventiva obbligatoria (10 secondi) per distribuire i token nel piano gratuito di Groq
        time.sleep(10)
    else:
        print(f"\n[Salto File] Impossibile elaborare {nome_file} dopo 5 tentativi.")

print(f"\nProcesso completato! Il file finale pulito è salvato in: '{OUTPUT_CSV}'")


Trovati 497 file da elaborare...
Avvio estrazione tramite Llama via Groq API...


100%|██████████| 5/5 [00:55<00:00, 11.03s/it]


Processo completato! Il file finale pulito è salvato in: 'schede_estratte_groq_llama.csv'


In [ ]:
df_groq2 = pd.read_csv('/content/schede_estratte_groq_llama.csv')
df_groq['scheda_llm'][1]

'The population is experiencing a period of relative improvement in food security, with a notable decline in the number of people facing severe food insecurity, from nearly 1.9 million to approximately 1.2 million, which represents about 10 percent of the total population, and are projected to be in a state of crisis, as indicated by technical food security indicators, during the harvest period, which is likely attributed to favourable agricultural performance and abundant rainfall, as well as an increase in household food stocks, although the prices of essential items are expected to remain elevated due to high transportation costs, which is a recurrent shock that affects the overall well-being of the affected areas, where the combination of structural drivers and technical indicators suggests a complex and dynamic situation, with humanitarian impacts that are still being felt, despite the slight improvement, highlighting the need for continued support and monitoring to address the un

In [ ]:
import os
import re
import time
import pandas as pd
from tqdm import tqdm
from groq import Groq

# 1. Configurazione del client Groq (Incolla qui la tua API Key di Groq)
client = Groq(api_key=GROQ_API_KEY)

# Usiamo Llama 3.3 70B: incredibilmente potente e preciso sui vincoli logici
MODELLO_LLAMA = "llama-3.3-70b-versatile"

OUTPUT_CSV = "schede_estratte_groq_llama2.csv"

files = [f for f in os.listdir(TESTO_DIR) if f.endswith(".txt")]
print(f"Trovati {len(files)} file da elaborare...")

# Prompt blindati per testo fluido senza etichette o rimasugli geografici
# PROMPT DI SISTEMA: Focalizzato sul ruolo e sulle regole di anonimizzazione
prompt_sistema = (
    "You are an expert humanitarian data analyst. Your job is to extract data from food insecurity reports into an anonymous summary.\n\n"
    "SAFETY CONSTRAINTS:\n"
    "- Completely eliminate all names of countries, cities, regions, provinces, or dates (months/years).\n"
    "- Do NOT use placeholders or tags like '[Country]' or '[Region]'. Use generic words like 'the affected areas', 'the population', or 'the country'.\n"
    "- You MUST preserve all exact numbers, percentages, figures, and technical classifications (e.g., IPC Phase 3, IPC Phase 4, 1.2 million, 10 percent)."
)

# PROMPT DI STRUTTURA: Guida il modello passo-passo sulla forma dell'output (One-Shot vincolante)
prompt_struttura = (
    "Analyze the provided report and output EXACTLY a plain bulleted list with 4 points. "
    "Do not include any introduction, introduction sentence, or markdown headers. Start directly with the first bullet point.\n\n"
    "Follow this exact structure:\n"
    "* [Drivers & Economy]: Summarize structural drivers, agricultural performance, and economic factors (prices/transportation) without places or dates.\n"
    "* [Current Data]: State the exact technical indicators for the CURRENT period (numbers of people, percentages, and IPC phases) in a standalone sentence.\n"
    "* [Projected Data]: State the exact technical indicators for the PROJECTED period (numbers of people, percentages, and IPC phases) in a standalone sentence.\n"
    "* [Humanitarian Impact]: Summarize impacts on livelihoods and nutrition.\n\n"
    "Report to analyze:\n"
)



print("Avvio estrazione tramite Llama via Groq API...")
risultati_finali = []

# Ciclo principale di elaborazione
for nome_file in tqdm(files[:5]):
    path = os.path.join(TESTO_DIR, nome_file)
    with open(path, "r", encoding="utf-8") as f:
        testo = f.read()

    # Pulizia del testo secondo la tua regex nativa
    testo_pulito = re.sub(r'^.*?={40}\n\n', '', testo, flags=re.DOTALL).strip()
    if not testo_pulito:
        continue

    successo = False
    tentativi = 0
    scheda_anonima = ""

    # Gestione automatica dei limiti di token al minuto (TPM) del piano free di Groq
    while not successo and tentativi < 5:
        try:
            completion = client.chat.completions.create(
                model=MODELLO_LLAMA,
                messages=[
                    {"role": "system", "content": prompt_sistema},
                    {"role": "user", "content": f"{prompt_struttura}\n\nReport:\n{testo_pulito}"}
                ],
                temperature=0.1,
                max_tokens=500
            )

            # --- ESTRAZIONE STANDARD CORRETTA E SICURA ---
            scheda_anonima = completion.choices[0].message.content.strip()
            successo = True

        except Exception as e:
            errore_str = str(e)
            # Rilevamento dei limiti di velocità (Quota API esaurita al minuto)
            if "429" in errore_str or "rate_limit_exceeded" in errore_str or "RESOURCE_EXHAUSTED" in errore_str:
                print(f"\n[Freno di Sicurezza] Limite di Token raggiunto sul file {nome_file}. Attendo 65 secondi...")
                time.sleep(65)
                tentativi += 1
            else:
                print(f"\nErrore imprevisto sul file {nome_file}: {e}")
                tentativi += 1
                time.sleep(5)

    if successo:
        # Rimozione di sicurezza finale di eventuali anomalie grafiche (es. parentesi quadre)
        scheda_anonima = re.sub(r'\[.*?\]', '', scheda_anonima).strip()
        scheda_anonima = re.sub(r'###.*?\n', '', scheda_anonima).strip()

        # Gestione metadati nativa
        metadata = estrai_metadata(nome_file) if 'estrai_metadata' in globals() else nome_file

        # Salvataggio incrementale nel CSV ad ogni iterazione
        nuovo_dato = pd.DataFrame([{
            "file": nome_file,
            "metadata": metadata,
            "testo_originale": testo_pulito,
            "scheda_llm": scheda_anonima
        }])

        nuovo_dato.to_csv(OUTPUT_CSV, mode='a', index=False, header=not os.path.exists(OUTPUT_CSV), encoding='utf-8')

        # Pausa preventiva obbligatoria (10 secondi) per distribuire i token nel piano gratuito di Groq
        time.sleep(10)
    else:
        print(f"\n[Salto File] Impossibile elaborare {nome_file} dopo 5 tentativi.")

print(f"\nProcesso completato! Il file finale pulito è salvato in: '{OUTPUT_CSV}'")


Trovati 497 file da elaborare...
Avvio estrazione tramite Llama via Groq API...


100%|██████████| 5/5 [00:54<00:00, 10.88s/it]


Processo completato! Il file finale pulito è salvato in: 'schede_estratte_groq_llama2.csv'


In [ ]:
df_groq3 = pd.read_csv('/content/schede_estratte_groq_llama2.csv')
df_groq3['scheda_llm'][1]

'* : The expected favourable agricultural performance, abundant rainfall, and increase in household food stocks are likely to improve the situation, although the prices of manufactured products are expected to remain higher than average due to the high cost of transportation.\n* : 1.9 million people are currently classified in IPC Phase 3 or above (Crisis or worse).\n* : Nearly 1.2 million people (10 percent of the total population analysed) are projected to be in IPC Phase 3 (Crisis).\n* : The situation affects the livelihoods and nutrition of the population, with significant numbers of people facing food insecurity, although the expected improvement may alleviate some of the humanitarian concerns.'

In [ ]:
df_groq3['testo_originale'][1]

'Between January and March 2025, which coincides with the harvest period, nearly 1.2 million people (10 percent of the total population analysed) are projected to be in IPC Phase 3 (Crisis). This is a marked improvement from the current period (November to December 2024), where 1.9 million people were classified in IPC Phase 3 or above (Crisis or worse). \nThe improvement is likely a result of the expected favourable agricultural performance and abundant rainfall, as well as the increase in household food stocks, even if during this period, the prices of manufactured products are expected to remain higher than average due to the high cost transportation.'

In [ ]:
import os
import re
import time
import pandas as pd
from tqdm import tqdm
from groq import Groq

# 1. Configurazione del client Groq (Incolla qui la tua API Key di Groq)

client = Groq(api_key=GROQ_API_KEY)

# Usiamo Llama 3.3 70B: incredibilmente potente e preciso sui vincoli logici
MODELLO_LLAMA = "llama-3.3-70b-versatile"

OUTPUT_CSV = "schede_estratte_groq_llama3.csv"

files = [f for f in os.listdir(TESTO_DIR) if f.endswith(".txt")]
print(f"Trovati {len(files)} file da elaborare...")

# Prompt blindati per testo fluido senza etichette o rimasugli geografici
# PROMPT DI SISTEMA AGGIORNATO CON TAG SEMANTICI PER EMBEDDING
prompt_sistema = (
    "You are an expert humanitarian data analyst. Your job is to rewrite food insecurity reports into an anonymous summary optimized for vector embeddings.\n"
    "CRITICAL RULE 1: Absolute Anonymization. You must completely eliminate all names of countries, cities, regions, provinces, or dates (months/years).\n"
    "CRITICAL RULE 2: No Placeholders. Do NOT use tags like '[Country]' or '[Region]'. Use generic words like 'the affected areas', 'the population', or 'the country'.\n"
    "CRITICAL RULE 3: Exact Data Integrity. You MUST preserve all exact numbers, percentages, figures, and technical classifications (e.g., IPC Phase 3, 1.2 million, 10 percent). Do NOT convert numbers into generic words.\n"
    "CRITICAL RULE 4: Strict Section Tagging. You must structure your output using exactly 4 plain lines. Each line MUST start with the specific semantic tag provided, followed by a colon and the text. Do not use markdown headers (like ###) or bullet points (*).\n"
    "The 4 required lines must start exactly with:\n"
    "- [DRIVERS AND ECONOMIC FACTORS]: \n"
    "- [CURRENT FOOD SECURITY DATA]: \n"
    "- [PROJECTED FOOD SECURITY DATA]: \n"
    "- [HUMANITARIAN IMPACTS]: "
)

prompt_struttura = (
    "Analyze the following report and write a clear, anonymous summary structured into the 4 requested tagged lines. "
    "Maintain every single numerical figure, percentage, and IPC phase exactly as written, but anonymize all places and dates:\n"
    "1) Line 1 must start with [DRIVERS AND ECONOMIC FACTORS]: and focus on drivers, agriculture, and economic factors.\n"
    "2) Line 2 must start with [CURRENT FOOD SECURITY DATA]: and list exact technical indicators for the current period.\n"
    "3) Line 3 must start with [PROJECTED FOOD SECURITY DATA]: and list exact technical indicators for the projected period.\n"
    "4) Line 4 must start with [HUMANITARIAN IMPACTS]: and detail the humanitarian impacts on livelihoods and nutrition.\n\n"
    "Report to analyze:\n"
)




print("Avvio estrazione tramite Llama via Groq API...")
risultati_finali = []

# Ciclo principale di elaborazione
for nome_file in tqdm(files[:5]):
    path = os.path.join(TESTO_DIR, nome_file)
    with open(path, "r", encoding="utf-8") as f:
        testo = f.read()

    # Pulizia del testo secondo la tua regex nativa
    testo_pulito = re.sub(r'^.*?={40}\n\n', '', testo, flags=re.DOTALL).strip()
    if not testo_pulito:
        continue

    successo = False
    tentativi = 0
    scheda_anonima = ""

    # Gestione automatica dei limiti di token al minuto (TPM) del piano free di Groq
    while not successo and tentativi < 5:
        try:
            completion = client.chat.completions.create(
                model=MODELLO_LLAMA,
                messages=[
                    {"role": "system", "content": prompt_sistema},
                    {"role": "user", "content": f"{prompt_struttura}\n\nReport:\n{testo_pulito}"}
                ],
                temperature=0.1,
                max_tokens=500
            )

            # --- ESTRAZIONE STANDARD CORRETTA E SICURA ---
            scheda_anonima = completion.choices[0].message.content.strip()
            successo = True

        except Exception as e:
            errore_str = str(e)
            # Rilevamento dei limiti di velocità (Quota API esaurita al minuto)
            if "429" in errore_str or "rate_limit_exceeded" in errore_str or "RESOURCE_EXHAUSTED" in errore_str:
                print(f"\n[Freno di Sicurezza] Limite di Token raggiunto sul file {nome_file}. Attendo 65 secondi...")
                time.sleep(65)
                tentativi += 1
            else:
                print(f"\nErrore imprevisto sul file {nome_file}: {e}")
                tentativi += 1
                time.sleep(5)

    if successo:
        # Rimozione di sicurezza finale di eventuali anomalie grafiche (es. parentesi quadre)
        scheda_anonima = re.sub(r'\[.*?\]', '', scheda_anonima).strip()
        scheda_anonima = re.sub(r'###.*?\n', '', scheda_anonima).strip()

        # Gestione metadati nativa
        metadata = estrai_metadata(nome_file) if 'estrai_metadata' in globals() else nome_file

        # Salvataggio incrementale nel CSV ad ogni iterazione
        nuovo_dato = pd.DataFrame([{
            "file": nome_file,
            "metadata": metadata,
            "testo_originale": testo_pulito,
            "scheda_llm": scheda_anonima
        }])

        nuovo_dato.to_csv(OUTPUT_CSV, mode='a', index=False, header=not os.path.exists(OUTPUT_CSV), encoding='utf-8')

        # Pausa preventiva obbligatoria (10 secondi) per distribuire i token nel piano gratuito di Groq
        time.sleep(10)
    else:
        print(f"\n[Salto File] Impossibile elaborare {nome_file} dopo 5 tentativi.")

print(f"\nProcesso completato! Il file finale pulito è salvato in: '{OUTPUT_CSV}'")

Trovati 497 file da elaborare...
Avvio estrazione tramite Llama via Groq API...


100%|██████████| 5/5 [00:55<00:00, 11.13s/it]


Processo completato! Il file finale pulito è salvato in: 'schede_estratte_groq_llama3.csv'


In [ ]:
df_groq4 = pd.read_csv('/content/schede_estratte_groq_llama3.csv')
df_groq4['scheda_llm'][1]

': The affected areas are expected to experience favourable agricultural performance and abundant rainfall, which is likely to improve household food stocks, despite the prices of manufactured products remaining higher than average due to the high cost of transportation.\n: In the current period, 1.9 million people are classified in IPC Phase 3 or above (Crisis or worse), which accounts for a significant portion of the population.\n: Nearly 1.2 million people (10 percent of the total population analysed) are projected to be in IPC Phase 3 (Crisis), indicating a marked improvement from the current period.\n: The humanitarian impacts on livelihoods and nutrition are significant, with a substantial number of people relying on limited food resources, and the situation is expected to affect the well-being of 1.2 million people, highlighting the need for continued support to address food insecurity.'

In [ ]:
import os
import re
import time
import pandas as pd
from tqdm import tqdm
from groq import Groq

# 1. Configurazione (Ricordati di usare la NUOVA chiave revocando la vecchia)
client = Groq(api_key=GROQ_API_KEY)

MODELLO_LLAMA = "llama-3.3-70b-versatile"

# Percorsi dei file su Google Drive
CSV_INPUT_UNITI = "/content/drive/MyDrive/HERO/dataset_report_uniti.csv"
OUTPUT_CSV_ANONIMO = "/content/drive/MyDrive/HERO/dataset_report_anonimizzati.csv"

# Caricamento del dataset unificato
if os.path.exists(CSV_INPUT_UNITI):
    df_uniti = pd.read_csv(CSV_INPUT_UNITI)
    print(f"Caricato dataset unificato con {len(df_uniti)} righe.")
else:
    raise FileNotFoundError(f"Non ho trovato il file {CSV_INPUT_UNITI}.")

# --- PROMPT AGGIORNATI CON TUTTI I DRIVER E STRUTTURA FLUIDA ---
prompt_sistema = (
    "You are an expert humanitarian data analyst. Your job is to rewrite food insecurity reports into an anonymous summary optimized for vector embeddings.\n"
    "CRITICAL RULE 1: Absolute Anonymization. You must completely eliminate all names of countries, cities, regions, provinces, or dates (months/years).\n"
    "CRITICAL RULE 2: No Placeholders. Do NOT use tags like '[Country]' or '[Region]'. Use generic words like 'the affected areas', 'the population', or 'the country'.\n"
    "CRITICAL RULE 3: Exact Data Integrity. You MUST preserve all exact numbers, percentages, figures, and technical classifications (e.g., IPC Phase 3, 1.2 million, 10 percent). Do NOT convert numbers into generic words.\n"
    "CRITICAL RULE 4: Strict Section Tagging. You must structure your output using exactly 4 plain paragraphs. Each paragraph MUST start with the specific semantic tag provided, followed by a colon and the text. Do not use markdown headers (like ###) or bullet points (*).\n"
    "The 4 required tags are:\n"
    "- [SHOCKS AND DRIVERS]: \n"
    "- [CURRENT FOOD SECURITY DATA]: \n"
    "- [PROJECTED FOOD SECURITY DATA]: \n"
    "- [HUMANITARIAN IMPACTS]: "
)

prompt_struttura = (
    "Analyze the following report and write a clear, anonymous summary structured into the 4 requested tagged sections. "
    "Maintain every single numerical figure, percentage, and IPC phase exactly as written, but anonymize all places and dates:\n"
    "1) Section 1 must start with [SHOCKS AND DRIVERS]: and focus on ALL underlying causes: economic factors, agriculture, climate shocks (rainfall, droughts), conflict, displacement, and disease outbreaks.\n"
    "2) Section 2 must start with [CURRENT FOOD SECURITY DATA]: and list exact technical indicators for the current period.\n"
    "3) Section 3 must start with [PROJECTED FOOD SECURITY DATA]: and list exact technical indicators for the projected period.\n"
    "4) Section 4 must start with [HUMANITARIAN IMPACTS]: and detail the humanitarian impacts on livelihoods, nutrition, and displacement consequences.\n\n"
    "Report to analyze:\n"
)

print("Avvio anonimizzazione tramite Llama via Groq API...")

# Ciclo principale sulle righe del DataFrame
for index, row in tqdm(df_uniti.iterrows(), total=len(df_uniti), desc="Elaborazione report"):

    # Controllo per riprendere il lavoro in caso di crash
    if os.path.exists(OUTPUT_CSV_ANONIMO):
        df_check = pd.read_csv(OUTPUT_CSV_ANONIMO)
        if row['folder_name'] in df_check['folder_name'].values:
            continue

    testo_pulito = row['text']
    if pd.isna(testo_pulito) or not str(testo_pulito).strip():
        continue

    successo = False
    tentativi = 0
    scheda_anonima = ""

    while not successo and tentativi < 5:
        try:
            completion = client.chat.completions.create(
                model=MODELLO_LLAMA,
                messages=[
                    {"role": "system", "content": prompt_sistema},
                    {"role": "user", "content": f"{prompt_struttura}\n\nReport:\n{testo_pulito}"}
                ],
                temperature=0.1,
                max_tokens=800  # Spazio sufficiente per un testo unito e dettagliato
            )
            scheda_anonima = completion.choices[0].message.content.strip()
            successo = True

        except Exception as e:
            errore_str = str(e)
            if "429" in errore_str or "rate_limit_exceeded" in errore_str or "RESOURCE_EXHAUSTED" in errore_str:
                print(f"\n[Freno di Sicurezza] Rate limit raggiunto. Attendo 65 secondi...")
                time.sleep(65)
                tentativi += 1
            else:
                print(f"\nErrore imprevisto alla riga {index}: {e}")
                tentativi += 1
                time.sleep(5)

    if successo:
        # Pulizia di sicurezza solo per rimasugli di markdown di intestazione (es. ###)
        # NON tocchiamo le parentesi quadre dei tag [SHOCKS AND DRIVERS] per non rovinare la struttura semantica del testo unito
        scheda_anonima = re.sub(r'###.*?\n', '', scheda_anonima).strip()

        # Creazione della riga con il testo unito pronto per l'embedder
        nuovo_dato = pd.DataFrame([{
            "folder_name": row['folder_name'],
            "country": row['country'],
            "start_period": row['start_period'],
            "end_period": row['end_period'],
            "language": row['language'],
            "testo_originale_unito": testo_pulito,
            "report_anonimo_unito": scheda_anonima  # <--- Questa è la colonna che manderai all'embedder
        }])

        # Salvataggio incrementale nel CSV ad ogni iterazione
        nuovo_dato.to_csv(OUTPUT_CSV_ANONIMO, mode='a', index=False, header=not os.path.exists(OUTPUT_CSV_ANONIMO), encoding='utf-8')

        # Pausa preventiva obbligatoria per il piano gratuito di Groq
        time.sleep(10)
    else:
        print(f"\n[Salto Riga] Impossibile elaborare l'indice {index} dopo 5 tentativi.")

print(f"\nProcesso completato! Il file pronto per l'embedder è: '{OUTPUT_CSV_ANONIMO}'")


In [ ]:
import os
import re
import time
import pandas as pd
from tqdm import tqdm
from groq import Groq

# 1. CONFIGURAZIONE CHIAVE API (Usa la nuova chiave generata nella console)
client = Groq(api_key=GROQ_API_KEY)

# Llama 3.3 70B: la scelta migliore per non perdere i dati numerici e le logiche di anonimizzazione
#MODELLO_LLAMA = "llama-3.3-70b-versatile"
MODELLO_LLAMA = "openai/gpt-oss-20b"

# Percorsi dei file su Google Drive
CSV_INPUT_UNITI = "/content/drive/MyDrive/HERO/dataset_report_uniti.csv"
OUTPUT_CSV_ANONIMO = "/content/drive/MyDrive/HERO/dataset_report_anonimizzati.csv"

# Assicuriamoci che il Drive sia montato prima di procedere
from google.colab import drive
drive.mount('/content/drive')

# Caricamento del dataset unificato
if os.path.exists(CSV_INPUT_UNITI):
    df_uniti = pd.read_csv(CSV_INPUT_UNITI)
    print(f"Caricato dataset unificato con {len(df_uniti)} righe. Pronto per Groq.")
else:
    raise FileNotFoundError(f"Non ho trovato il file {CSV_INPUT_UNITI}. Verifica i passaggi precedenti.")

# --- PROMPT OTTIMIZZATI CON TUTTI I DRIVER E STRUTTURA FLUIDA ---
prompt_sistema = (
    "You are an expert humanitarian data analyst. Your job is to rewrite food insecurity reports into an anonymous summary optimized for vector embeddings.\n"
    "CRITICAL RULE 1: Absolute Anonymization. You must completely eliminate all names of countries, cities, regions, provinces, or dates (months/years).\n"
    "CRITICAL RULE 2: No Placeholders. Do NOT use tags like '[Country]' or '[Region]'. Use generic words like 'the affected areas', 'the population', or 'the country'.\n"
    "CRITICAL RULE 3: Exact Data Integrity. You MUST preserve all exact numbers, percentages, figures, and technical classifications (e.g., IPC Phase 3, 1.2 million, 10 percent). Do NOT convert numbers into generic words.\n"
    "CRITICAL RULE 4: Strict Section Tagging. You must structure your output using exactly 4 plain paragraphs. Each paragraph MUST start with the specific semantic tag provided, followed by a colon and the text. Do not use markdown headers (like ###) or bullet points (*).\n"
    "CRITICAL RULE 5: Output Language. You MUST write the entire output in English, even if the input report is written in French, Spanish, or any other language.\n"
    "The 4 required tags are:\n"
    "- [SHOCKS AND DRIVERS]: \n"
    "- [CURRENT FOOD SECURITY DATA]: \n"
    "- [PROJECTED FOOD SECURITY DATA]: \n"
    "- [HUMANITARIAN IMPACTS]: "
)

prompt_struttura = (
    "Analyze the following report and write a clear, anonymous summary structured into the 4 requested tagged sections. "
    "Maintain every single numerical figure, percentage, and IPC phase exactly as written, but anonymize all places and dates:\n"
    "1) Section 1 must start with [SHOCKS AND DRIVERS]: and focus on ALL underlying causes: economic factors, agriculture, climate shocks (rainfall, droughts), conflict, displacement, and disease outbreaks.\n"
    "2) Section 2 must start with [CURRENT FOOD SECURITY DATA]: and list exact technical indicators for the current period.\n"
    "3) Section 3 must start with [PROJECTED FOOD SECURITY DATA]: and list exact technical indicators for the projected period.\n"
    "4) Section 4 must start with [HUMANITARIAN IMPACTS]: and detail the humanitarian impacts on livelihoods, nutrition, and displacement consequences.\n\n"
    "Report to analyze:\n"
)

print("\nAvvio anonimizzazione tramite Llama 3.3 70B via Groq API...")

# Ciclo principale sulle righe del DataFrame
for index, row in tqdm(df_uniti.iterrows(), total=len(df_uniti), desc="Elaborazione report"):

    # MECCANISMO DI PERSISTENZA: Se Colab si disconnette, salta le righe già fatte salvate nel CSV
    if os.path.exists(OUTPUT_CSV_ANONIMO):
        df_check = pd.read_csv(OUTPUT_CSV_ANONIMO)
        if row['folder_name'] in df_check['folder_name'].values:
            continue

    testo_pulito = row['text']
    if pd.isna(testo_pulito) or not str(testo_pulito).strip():
        continue

    successo = False
    tentativi = 0
    scheda_anonima = ""

    # Ciclo di gestione degli errori e del rate limit (Fino a 5 tentativi per report)
    while not successo and tentativi < 5:
        try:
            completion = client.chat.completions.create(
                model=MODELLO_LLAMA,
                messages=[
                    {"role": "system", "content": prompt_sistema},
                    {"role": "user", "content": f"{prompt_struttura}\n\nReport:\n{testo_pulito}"}
                ],
                temperature=0.1,
                max_tokens=800  # Limite bilanciato per contenere i consumi del piano free
            )
            scheda_anonima = completion.choices[0].message.content.strip()
            successo = True

        except Exception as e:
            errore_str = str(e)
            # Se colpiamo il limite di token al minuto (TPM), attiviamo il freno di emergenza di 65 secondi
            if "429" in errore_str or "rate_limit_exceeded" in errore_str or "RESOURCE_EXHAUSTED" in errore_str:
                print(f"\n[Freno Emergenza 429] Limite raggiunto alla riga {index}. Attendo 65 secondi per svuotare il contatore...")
                time.sleep(65)
                tentativi += 1
            else:
                print(f"\nErrore imprevisto alla riga {index}: {e}")
                tentativi += 1
                time.sleep(5)

    if successo:
        # Pulizia di sicurezza solo per rimosso markdown di formattazione pesante (###)
        # NOTA: Manteniamo le parentesi quadre perché servono come ancore semantiche per l'embedder
        scheda_anonima = re.sub(r'###.*?\n', '', scheda_anonima).strip()

        # Creazione del nuovo record con il testo unito pronto per l'embedder
        nuovo_dato = pd.DataFrame([{
            "folder_name": row['folder_name'],
            "country": row['country'],
            "start_period": row['start_period'],
            "end_period": row['end_period'],
            "language": row['language'],
            "testo_originale_unito": testo_pulito,
            "report_anonimo_unito": scheda_anonima  # <-- Questa colonna andrà inviata al modello di embedding
        }])

        # Scrittura incrementale sul CSV su Drive (Massima sicurezza contro i crash di Colab)
        nuovo_dato.to_csv(OUTPUT_CSV_ANONIMO, mode='a', index=False, header=not os.path.exists(OUTPUT_CSV_ANONIMO), encoding='utf-8')

        # PAUSA STRATEGICA DI 40 SECONDI: Impedisce l'accumulo di token nel piano gratuito di Groq
        time.sleep(40)
    else:
        print(f"\n[Salto Riga] Impossibile elaborare l'indice {index} dopo 5 tentativi consecutivi.")

print(f"\nProcesso completato! Il file finale per gli embedding è salvato in: '{OUTPUT_CSV_ANONIMO}'")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Caricato dataset unificato con 492 righe. Pronto per Groq.

Avvio anonimizzazione tramite Llama 3.3 70B via Groq API...


Elaborazione report:   0%|          | 0/492 [00:00<?, ?it/s]


[Freno Emergenza 429] Limite raggiunto alla riga 1. Attendo 65 secondi per svuotare il contatore...

[Freno Emergenza 429] Limite raggiunto alla riga 1. Attendo 65 secondi per svuotare il contatore...


Elaborazione report:   0%|          | 1/492 [01:46<14:31:26, 106.49s/it]


KeyboardInterrupt: 

In [ ]:
import os
import re
import time
import pandas as pd
from tqdm import tqdm
from google.colab import userdata

# Installa la libreria ufficiale se mancante
try:
    import google.generativeai as genai
except ImportError:
    !pip install -q google-generativeai
    import google.generativeai as genai

# 1. CONFIGURAZIONE CHIAVE API GEMINI
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
genai.configure(api_key=GEMINI_API_KEY)

# Usiamo Gemini 1.5 Flash: ideale per finestre di contesto enormi e piano gratuito generoso
model = genai.GenerativeModel('gemini-1.5-flash')

# Percorsi dei file su Google Drive
CSV_INPUT_UNITI = "/content/drive/MyDrive/HERO/dataset_report_uniti.csv"
OUTPUT_CSV_ANONIMO = "/content/drive/MyDrive/HERO/dataset_report_anonimizzati.csv"

# Assicuriamoci che il Drive sia montato
from google.colab import drive
drive.mount('/content/drive')

# Caricamento del dataset unificato
if os.path.exists(CSV_INPUT_UNITI):
    df_uniti = pd.read_csv(CSV_INPUT_UNITI)
    print(f"Caricato dataset unificato con {len(df_uniti)} righe. Pronto per Gemini.")
else:
    raise FileNotFoundError(f"Non ho trovato il file {CSV_INPUT_UNITI}.")

# --- PROMPT OTTIMIZZATI PER COMPRENDERE TUTTI I DRIVER ---
prompt_sistema = (
    "You are an expert humanitarian data analyst. Your job is to rewrite food insecurity reports into an anonymous summary optimized for vector embeddings.\n"
    "CRITICAL RULE 1: Absolute Anonymization. You must completely eliminate all names of countries, cities, regions, provinces, or dates (months/years).\n"
    "CRITICAL RULE 2: No Placeholders. Do NOT use tags like '[Country]' or '[Region]'. Use generic words like 'the affected areas', 'the population', or 'the country'.\n"
    "CRITICAL RULE 3: Exact Data Integrity. You MUST preserve all exact numbers, percentages, figures, and technical classifications (e.g., IPC Phase 3, 1.2 million, 10 percent). Do NOT convert numbers into generic words.\n"
    "CRITICAL RULE 4: Strict Section Tagging. You must structure your output using exactly 4 plain paragraphs. Each paragraph MUST start with the specific semantic tag provided, followed by a colon and the text. Do not use markdown headers (like ###) or bullet points (*).\n"
    "CRITICAL RULE 5: Output Language. You MUST write the entire output in English, even if the input report is written in French, Spanish, or any other language.\n"
    "The 4 required tags are:\n"
    "- [SHOCKS AND DRIVERS]: \n"
    "- [CURRENT FOOD SECURITY DATA]: \n"
    "- [PROJECTED FOOD SECURITY DATA]: \n"
    "- [HUMANITARIAN IMPACTS]: "
)

prompt_struttura = (
    "Analyze the following report and write a clear, anonymous summary structured into the 4 requested tagged sections. "
    "Maintain every single numerical figure, percentage, and IPC phase exactly as written, but anonymize all places and dates:\n"
    "1) Section 1 must start with [SHOCKS AND DRIVERS]: and focus on ALL underlying causes: economic factors, agriculture, climate shocks (rainfall, droughts), conflict, displacement, and disease outbreaks.\n"
    "2) Section 2 must start with [CURRENT FOOD SECURITY DATA]: and list exact technical indicators for the current period.\n"
    "3) Section 3 must start with [PROJECTED FOOD SECURITY DATA]: and list exact technical indicators for the projected period.\n"
    "4) Section 4 must start with [HUMANITARIAN IMPACTS]: and detail the humanitarian impacts on livelihoods, nutrition, and displacement consequences.\n\n"
    "Report to analyze:\n"
)

print("\nAvvio anonimizzazione tramite Gemini 1.5 Flash...")

# Ciclo principale sulle righe del DataFrame
for index, row in tqdm(df_uniti.iterrows(), total=len(df_uniti), desc="Elaborazione report"):

    # MECCANISMO DI PERSISTENZA: Salta le righe già elaborate nel CSV di output
    if os.path.exists(OUTPUT_CSV_ANONIMO):
        df_check = pd.read_csv(OUTPUT_CSV_ANONIMO)
        if row['folder_name'] in df_check['folder_name'].values:
            continue

    testo_pulito = row['text']
    if pd.isna(testo_pulito) or not str(testo_pulito).strip():
        continue

    successo = False
    tentativi = 0
    scheda_anonima = ""

    # Uniamo i prompt e il testo da analizzare per Gemini
    prompt_completo = f"{prompt_sistema}\n\n{prompt_struttura}\n\nReport:\n{testo_pulito}"

    while not successo and tentativi < 5:
        try:
            # Configurazione della temperatura bassa per massima precisione logica
            response = model.generate_content(
                prompt_completo,
                generation_config={"temperature": 0.1}
            )
            scheda_anonima = response.text.strip()
            successo = True

        except Exception as e:
            errore_str = str(e)
            if "429" in errore_str or "Quota" in errore_str:
                print(f"\n[Rate Limit Gemini] Attendo 30 secondi prima di riprovare la riga {index}...")
                time.sleep(30)
                tentativi += 1
            else:
                print(f"\nErrore imprevisto alla riga {index}: {e}")
                tentativi += 1
                time.sleep(5)

    if successo:
        # Pulizia di sicurezza per rimosso markdown di formattazione pesante (###)
        scheda_anonima = re.sub(r'###.*?\n', '', scheda_anonima).strip()

        # Creazione del record
        nuovo_dato = pd.DataFrame([{
            "folder_name": row['folder_name'],
            "country": row['country'],
            "start_period": row['start_period'],
            "end_period": row['end_period'],
            "language": row['language'],
            "testo_originale_unito": testo_pulito,
            "report_anonimo_unito": scheda_anonima
        }])

        # Scrittura incrementale sul CSV su Drive
        nuovo_dato.to_csv(OUTPUT_CSV_ANONIMO, mode='a', index=False, header=not os.path.exists(OUTPUT_CSV_ANONIMO), encoding='utf-8')

        # Pausa minima precauzionale di 4 secondi (Gemini Free consente circa 15 richieste RPM)
        time.sleep(4)
    else:
        print(f"\n[Salto Riga] Impossibile elaborare l'indice {index} dopo 5 tentativi consecutivi.")

print(f"\nProcesso completato! Il file finale per gli embedding è salvato in: '{OUTPUT_CSV_ANONIMO}'")


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Caricato dataset unificato con 492 righe. Pronto per Gemini.

Avvio anonimizzazione tramite Gemini 1.5 Flash...


Elaborazione report:   0%|          | 0/492 [00:00<?, ?it/s]WARNING:tornado.access:404 POST /v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2517.58ms



Errore imprevisto alla riga 1: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.



Errore imprevisto alla riga 1: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.


Elaborazione report:   0%|          | 1/492 [00:13<1:51:01, 13.57s/it]


KeyboardInterrupt: 

In [ ]:
!pip install -q huggingface_hub

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!pip install -q transformers accelerate bitsandbytes pandas tqdm huggingface_hub

In [ ]:
import os
import re
import gc
import time
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

# Forza il download accelerato anche per questa sessione Python
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# --- CONFIGURAZIONE MODELLO UFFICIALE META ---
NOME_MODELLO = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Configurazione BitsAndBytes per blindare il modello a 4-bit dentro la GPU T4
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# Svuotiamo la cache prima di iniziare per recuperare ogni singolo MB di RAM liberi
torch.cuda.empty_cache()
gc.collect()

print("Scaricamento accelerato e caricamento del modello ufficiale Meta...")
tokenizer = AutoTokenizer.from_pretrained(NOME_MODELLO)
model = AutoModelForCausalLM.from_pretrained(
    NOME_MODELLO,
    quantization_config=quantization_config,
    device_map="auto"
)

# Creazione della pipeline di generazione di testo
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Token di stop nativi di Llama 3.1 per bloccare immediatamente l'output a fine risposta
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

# --- CONFIGURAZIONE PERCORSI DRIVE ---
CSV_INPUT_UNITI = "/content/drive/MyDrive/HERO/dataset_report_uniti.csv"
OUTPUT_CSV_ANONIMO = "/content/drive/MyDrive/HERO/dataset_report_anonimizzati_locale.csv"

from google.colab import drive
drive.mount('/content/drive')

if os.path.exists(CSV_INPUT_UNITI):
    df_uniti = pd.read_csv(CSV_INPUT_UNITI)
    print(f"\nCaricato dataset unificato con {len(df_uniti)} righe. Inizio elaborazione...")
else:
    raise FileNotFoundError(f"Impossibile trovare il file {CSV_INPUT_UNITI}.")

# --- PROMPT DI STRUTTURAZIONE ED ESTENSIONE DEI DRIVER ---
prompt_sistema = (
    "You are an expert humanitarian data analyst. Your job is to rewrite food insecurity reports into an anonymous summary optimized for vector embeddings.\n"
    "CRITICAL RULE 1: Absolute Anonymization. You must completely eliminate all names of countries, cities, regions, provinces, or dates (months/years).\n"
    "CRITICAL RULE 2: No Placeholders. Do NOT use tags like '[Country]' or '[Region]'. Use generic words like 'the affected areas', 'the population', or 'the country'.\n"
    "CRITICAL RULE 3: Exact Data Integrity. You MUST preserve all exact numbers, percentages, figures, and technical classifications (e.g., IPC Phase 3, 1.2 million, 10 percent). Do NOT convert numbers into generic words.\n"
    "CRITICAL RULE 4: Strict Section Tagging. You must structure your output using exactly 4 plain paragraphs. Each paragraph MUST start with the specific semantic tag provided, followed by a colon and the text. Do not use markdown headers (like ###) or bullet points (*).\n"
    "CRITICAL RULE 5: Output Language. You MUST write the entire output in English, even if the input report is written in French, Spanish, or any other language.\n"
    "The 4 required tags are:\n"
    "- [SHOCKS AND DRIVERS]: \n"
    "- [CURRENT FOOD SECURITY DATA]: \n"
    "- [PROJECTED FOOD SECURITY DATA]: \n"
    "- [HUMANITARIAN IMPACTS]: "
)

prompt_struttura = (
    "Analyze the following report and write a clear, anonymous summary structured into the 4 requested tagged sections. "
    "Maintain every single numerical figure, percentage, and IPC phase exactly as written, but anonymize all places and dates:\n"
    "1) Section 1 must start with [SHOCKS AND DRIVERS]: and focus on ALL underlying causes: economic factors, agriculture, climate shocks (rainfall, droughts), conflict, displacement, and disease outbreaks.\n"
    "2) Section 2 must start with [CURRENT FOOD SECURITY DATA]: and list exact technical indicators for the current period.\n"
    "3) Section 3 must start with [PROJECTED FOOD SECURITY DATA]: and list exact technical indicators for the projected period.\n"
    "4) Section 4 must start with [HUMANITARIAN IMPACTS]: and detail the humanitarian impacts on livelihoods, nutrition, and displacement consequences.\n\n"
    "Report to analyze:\n"
)

# Ciclo principale di scorrimento del DataFrame unificato
for index, row in tqdm(df_uniti.iterrows(), total=len(df_uniti), desc="Elaborazione report"):

    # Persistenza: salta se già fatto
    if os.path.exists(OUTPUT_CSV_ANONIMO):
        df_check = pd.read_csv(OUTPUT_CSV_ANONIMO)
        if row['folder_name'] in df_check['folder_name'].values:
            continue

    testo_pulito = row['text']
    if pd.isna(testo_pulito) or not str(testo_pulito).strip():
        continue

    # Formattazione del prompt
    messages = [
        {"role": "system", "content": prompt_sistema},
        {"role": "user", "content": f"{prompt_struttura}\n\nReport:\n{testo_pulito}"}
    ]

    prompt_formattato = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    try:
        # Svuotamento preventivo aggressivo della cache VRAM prima della chiamata
        torch.cuda.empty_cache()
        gc.collect()

        # Usiamo inference_mode per abbattere l'uso della RAM di calcolo ed evitare l'OOM
        with torch.inference_mode():
            outputs = pipe(
                prompt_formattato,
                max_new_tokens=700,      # Ottimizzato per contenere lo spazio di generazione
                do_sample=False,         # Massima precisione logica
                eos_token_id=terminators,
                pad_token_id=tokenizer.eos_token_id,
                return_full_text=False   # CRUCIALE: non alloca memoria per restituire il testo in ingresso
            )

        # Con return_full_text=False la stringa restituita contiene solo l'output dell'LLM
        scheda_anonima = outputs[0]["generated_text"].strip()
        scheda_anonima = re.sub(r'###.*?\n', '', scheda_anonima).strip()

        # Compilazione del record finale
        nuovo_dato = pd.DataFrame([{
            "folder_name": row['folder_name'],
            "country": row['country'],
            "start_period": row['start_period'],
            "end_period": row['end_period'],
            "language": row['language'],
            "testo_originale_unito": testo_pulito,
            "report_anonimo_unito": scheda_anonima
        }])

        # Scrittura sul CSV di sicurezza
        nuovo_dato.to_csv(OUTPUT_CSV_ANONIMO, mode='a', index=False, header=not os.path.exists(OUTPUT_CSV_ANONIMO), encoding='utf-8')

    except Exception as e:
        print(f"\nErrore riscontrato all'indice {index}: {e}")
        time.sleep(5)

print(f"\nProcesso completato! Il file finale pulito è in: '{OUTPUT_CSV_ANONIMO}'")



Scaricamento accelerato e caricamento del modello ufficiale Meta...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Caricato dataset unificato con 492 righe. Inizio elaborazione...


Elaborazione report:   0%|          | 0/492 [00:00<?, ?it/s][transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'pad_token_id', 'eos_token_id', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


Errore riscontrato all'indice 7: CUDA out of memory. Tried to allocate 92.18 GiB. GPU 0 has a total capacity of 14.56 GiB of which 2.12 GiB is free. Including non-PyTorch memory, this process has 12.45 GiB memory in use. Of the allocated memory 11.30 GiB is allocated by PyTorch, and 1.02 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)


Elaborazione report:   2%|▏         | 8/492 [00:44<44:46,  5.55s/it][transformers] Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Errore riscontrato all'indice 8: CUDA out of memory. Tried to allocate 97.75 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.78 GiB is free. Including non-PyTorch memory, this process has 12.78 GiB memory in use. Of the allocated memory 11.56 GiB is allocated by PyTorch, and 1.09 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)


Elaborazione report:   2%|▏         | 8/492 [00:47<47:32,  5.89s/it]


KeyboardInterrupt: 